# AI-assisted topology synthesis: piezoelectric energy-harvester interface

Gemma 4 31B proposes interface circuits (connectivity **and** component values) for a piezoelectric
harvester; LTspice simulates each one under six operating conditions; the measurements are checked
against a fixed specification and fed back for the next proposal. Between model calls, a numerical
optimiser tunes the component values of the most promising design. When the search stops, the
selected design and the full-bridge-rectifier reference are checked on four **held-out** conditions
that the model never saw.

**Specification (every condition, 0.8 s cold start):** output reaches and stays at 1.8 V or more,
never exceeds 3.6 V, ripple at most 50 mV, reaches 1.8 V within 0.4 s, settled. Conditions vary
the load (150 kOhm to 2.2 MOhm), excitation (30-100 uA), frequency (80 Hz), piezo capacitance (+20 %)
and temperature (60 C). See section 1 of `report.md` once a run finishes.

**Run the cells in order:**
1. *Setup* (always).
2. *Package files*: writes `piezo_synth/` next to this notebook (always, after any update of this notebook).
3. *Settings* for the search.
4. *Smoke test* (first time only, about 2 minutes): simulates the reference and a known-good witness
   circuit on all 10 conditions, without the model. It checks LTspice and shows how hard the spec is.
5. *Full experiment* (typically 30-60 minutes): runs as a cluster job, so you can close the laptop.
   `log()` follows its output; stopping `log()` does **not** stop the job (`cancel(job)` does).
6. *Progress* (any time), *Results*, *Figures*, *Waveforms*.

Progress lines are tagged by phase: `[model]` waiting for Gemma, `[refine]` tuning values,
`[sim]` simulating, `[eval]` checking results, `[heldout]` final held-out check, `[done]`.


## 1. Setup

In [ ]:
import os, json, time, glob
from pathlib import Path
from gpu import run, log, jobs, cancel

os.makedirs("piezo_synth", exist_ok=True)
os.makedirs("runs", exist_ok=True)
NAMESPACE = open("/var/run/secrets/kubernetes.io/serviceaccount/namespace").read().strip()
LTSPICE = f"image-registry.openshift-image-registry.svc:5000/{NAMESPACE}/ltspice-cpu:latest"
RUN_ID = time.strftime("%Y%m%d-%H%M%S")   # re-run this cell for a new run id
print("LTspice image:", LTSPICE)
print("this run will write to runs/" + RUN_ID)

## 2. Package files

The next cells write the experiment's code into `piezo_synth/` beside this notebook (cells are
collapsed; expand to read). The job reads them from there. The specification lives in
`piezo_synth/spec.py`.

In [ ]:
%%writefile piezo_synth/__init__.py
"""AI-assisted topology synthesis for piezoelectric energy-harvesting interface circuits."""
__version__ = "0.1.0"


In [ ]:
%%writefile piezo_synth/spec.py
"""The experiment specification: operating conditions, pass/fail criteria, limits, simulation settings.

Search conditions (S*) are shown to the model and used for scoring. Held-out conditions (H*) are
evaluated only on the selected design and the reference, after the search has stopped.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import List


@dataclass(frozen=True)
class Condition:
    id: str
    split: str          # "search" | "heldout"
    label: str
    ip: float           # piezo current amplitude, A peak
    freq: float         # excitation frequency, Hz
    cp: float           # piezo capacitance, F
    rp: float           # piezo leakage resistance, ohm
    rl: float           # load resistance, ohm
    temp: float         # simulation temperature, degC


@dataclass(frozen=True)
class Criterion:
    id: str
    meas: str           # measurement name (see netlist.MEAS_NAMES, plus derived "dv")
    op: str             # ">=" | "<="
    limit: float
    scale: float        # normalises margins: margin = (meas - limit)/scale for ">="
    unit: str
    description: str


RP = 20e6

CONDITIONS: List[Condition] = [
    Condition("S1", "search", "nominal", 50e-6, 100, 25e-9, RP, 330e3, 27),
    Condition("S2", "search", "heavy load", 50e-6, 100, 25e-9, RP, 150e3, 27),
    Condition("S3", "search", "strong excitation, light load", 100e-6, 100, 25e-9, RP, 2.2e6, 27),
    Condition("S4", "search", "weak excitation", 30e-6, 100, 25e-9, RP, 1e6, 27),
    Condition("S5", "search", "low frequency", 40e-6, 80, 25e-9, RP, 470e3, 27),
    Condition("S6", "search", "Cp +20 %, hot", 50e-6, 100, 30e-9, RP, 330e3, 60),
    Condition("H1", "heldout", "weak excitation, mid load", 35e-6, 100, 25e-9, RP, 470e3, 27),
    Condition("H2", "heldout", "high frequency, Cp −15 %", 60e-6, 120, 21.25e-9, RP, 220e3, 27),
    Condition("H3", "heldout", "very strong excitation, open load, cold", 120e-6, 100, 25e-9, RP, 10e6, -20),
    Condition("H4", "heldout", "hot, heavy load", 50e-6, 100, 25e-9, RP, 200e3, 85),
]

CRITERIA: List[Criterion] = [
    Criterion("vout_min", "vavg", ">=", 1.8, 1.8, "V", "settled mean output reaches the load's minimum supply"),
    Criterion("vout_max", "vmax", "<=", 3.6, 3.6, "V", "output never exceeds the load's absolute maximum"),
    Criterion("ripple", "vpp", "<=", 0.05, 0.05, "V", "peak-to-peak ripple in the final window"),
    Criterion("startup", "tup", "<=", 0.4, 0.4, "s", "cold start: first time output reaches 1.8 V"),
    Criterion("settled", "dv", "<=", 0.02, 0.02, "V", "mean output changes little between the last two windows"),
]

LIMITS = {
    "max_components": 16,
    "max_switches": 4,
    "max_sync": 2,
    "R": (1.0, 1e8),
    "C": (1e-10, 47e-6),
    "L": (1e-5, 1e-2),
    "ton": (1e-7, 1e-3),
    "phase": (-0.25, 0.25),
    "l_rser_base": 2.0,
    "l_rser_per_h": 5000.0,
}

SIM = {"tstop": 0.8, "maxstep": 2e-6, "win": (0.7, 0.8), "win_prev": (0.6, 0.7)}

CTRL_SINK_A = 0.5e-6   # supply current charged per SYNC controller, drawn from "out"


def search_conditions() -> List[Condition]:
    return [c for c in CONDITIONS if c.split == "search"]


def heldout_conditions() -> List[Condition]:
    return [c for c in CONDITIONS if c.split == "heldout"]


def condition(cid: str) -> Condition:
    for c in CONDITIONS:
        if c.id == cid:
            return c
    raise KeyError(cid)


def _jsonable(v):
    return list(v) if isinstance(v, tuple) else v


def spec_dict() -> dict:
    return {
        "conditions": [asdict(c) for c in CONDITIONS],
        "criteria": [asdict(c) for c in CRITERIA],
        "limits": {k: _jsonable(v) for k, v in LIMITS.items()},
        "sim": {k: _jsonable(v) for k, v in SIM.items()},
        "ctrl_sink_a": CTRL_SINK_A,
    }


In [ ]:
%%writefile piezo_synth/circuit.py
"""Circuit representation proposed by the model: parse, validate, canonicalise, hash, edit values.

A circuit is a list of components over a fixed vocabulary, connected through named nodes.
Reserved nodes: "0" (ground), "pzp"/"pzn" (piezo terminals), "out" (storage/load node).
The test bench (piezo model, load, controller supply sinks) is added by netlist.py.
"""
from __future__ import annotations

import copy
import hashlib
import re
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

from .spec import LIMITS

ALLOWED_TYPES = ("R", "C", "L", "D", "S", "SYNC")
DIODE_MODELS = ("DSCH", "DSI", "DZ33")
RESERVED_NODES = ("0", "pzp", "pzn", "out")
MOVES = ("new_topology", "modify_topology", "resize")
EDGES = ("both", "pos", "neg")
TWO_TERMINAL = ("R", "C", "L", "D", "S")
VALUED = ("R", "C", "L")
# connections the harness itself makes to each reserved node (piezo model, load)
HARNESS_CONNECTIONS = {"pzp": 1, "pzn": 1, "out": 1}

_ID_RE = re.compile(r"^[A-Za-z][A-Za-z0-9_]{0,7}$")
_NODE_RE = re.compile(r"^[a-z][a-z0-9_]{0,11}$")
_SUFFIX = {"f": 1e-15, "p": 1e-12, "n": 1e-9, "u": 1e-6, "µ": 1e-6, "m": 1e-3,
           "k": 1e3, "meg": 1e6, "g": 1e9, "t": 1e12}


class CircuitError(ValueError):
    def __init__(self, problems: List[str]):
        super().__init__("; ".join(problems))
        self.problems = list(problems)


@dataclass
class Component:
    id: str
    type: str
    nodes: tuple = ()
    value: Optional[float] = None
    model: Optional[str] = None
    ctrl: Optional[str] = None
    ton: Optional[float] = None
    phase: Optional[float] = None
    edges: Optional[str] = None


@dataclass
class Circuit:
    name: str
    components: List[Component] = field(default_factory=list)
    move: str = "new_topology"
    rationale: str = ""
    done: bool = False


def parse_value(v) -> Optional[float]:
    """A number, or a SPICE-style string such as '2.2u', '4.7meg', '10k'. None if unparseable."""
    if isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        return float(v)
    if not isinstance(v, str):
        return None
    s = v.strip().lower().replace(" ", "")
    m =re.match(r"^([-+]?(?:\d+\.?\d*|\.\d+)(?:e[-+]?\d+)?)(meg|[fpnuµmkgt])?[a-z]*$", s)
    if not m:
        return None
    return float(m.group(1)) * (_SUFFIX[m.group(2)] if m.group(2) else 1.0)


# ---------------------------------------------------------------- parsing and validation

def parse_circuit(obj: dict) -> Circuit:
    """Build a Circuit from the model's JSON object; raise CircuitError listing every problem."""
    problems: List[str] = []
    if not isinstance(obj, dict):
        raise CircuitError(["the reply is not a JSON object"])
    comps_in = obj.get("components")
    if not isinstance(comps_in, list) or not comps_in:
        raise CircuitError(["'components' must be a non-empty list"])
    comps: List[Component] = []
    for k, raw in enumerate(comps_in):
        if not isinstance(raw, dict):
            problems.append(f"component #{k + 1} is not an object")
            continue
        cid = str(raw.get("id", f"#{k + 1}"))
        ctype = str(raw.get("type", "")).upper()
        nodes = raw.get("nodes", [])
        if not isinstance(nodes, list):
            problems.append(f"{cid}: 'nodes' must be a list")
            nodes = []
        num = {}
        for f in ("value", "ton", "phase"):
            if raw.get(f) is not None:
                num[f] = parse_value(raw[f])
                if num[f] is None:
                    problems.append(f"{cid}: {f}={raw[f]!r} is not a number")
        comps.append(Component(
            id=cid, type=ctype, nodes=tuple(str(n) for n in nodes),
            value=num.get("value"), model=(str(raw["model"]).upper() if raw.get("model") else None),
            ctrl=(str(raw["ctrl"]) if raw.get("ctrl") else None),
            ton=num.get("ton"), phase=num.get("phase"),
            edges=(str(raw["edges"]).lower() if raw.get("edges") else None)))
    c = Circuit(name=str(obj.get("name", "unnamed"))[:80], components=comps,
                move=str(obj.get("move", "new_topology")), rationale=str(obj.get("rationale", "")),
                done=bool(obj.get("done", False)))
    problems += validate(c)
    if problems:
        raise CircuitError(problems)
    return c


def _in(lo_hi, v) -> bool:
    return v is not None and lo_hi[0] <= v <= lo_hi[1]


def validate(c: Circuit) -> List[str]:
    """Every rule violation as a human-readable line naming the component."""
    p: List[str] = []
    if c.move not in MOVES:
        p.append(f"move={c.move!r} must be one of {', '.join(MOVES)}")
    ids = [x.id for x in c.components]
    for dup in sorted({i for i in ids if ids.count(i) > 1}):
        p.append(f"{dup}: duplicate id")
    syncs = {x.id for x in c.components if x.type == "SYNC"}
    used_syncs = set()
    for x in c.components:
        if not _ID_RE.match(x.id):
            p.append(f"{x.id}: id must be a letter followed by up to 7 letters/digits/_")
        if x.type not in ALLOWED_TYPES:
            p.append(f"{x.id}: type {x.type!r} is not allowed (use {', '.join(ALLOWED_TYPES)})")
            continue
        if x.type in TWO_TERMINAL:
            if len(x.nodes) != 2:
                p.append(f"{x.id}: needs exactly 2 nodes, got {len(x.nodes)}")
            elif x.nodes[0] == x.nodes[1]:
                p.append(f"{x.id}: both terminals on node {x.nodes[0]}")
            for n in x.nodes:
                if n != "0" and not _NODE_RE.match(n):
                    p.append(f"{x.id}: node name {n!r} must be lower-case, start with a letter, max 12 chars, or '0'")
        elif x.nodes:
            p.append(f"{x.id}: SYNC takes no nodes")
        if x.type in VALUED and not _in(LIMITS[x.type], x.value):
            lo, hi = LIMITS[x.type]
            p.append(f"{x.id}: value {x.value} outside [{lo:g}, {hi:g}]")
        if x.type == "D" and x.model not in DIODE_MODELS:
            p.append(f"{x.id}: diode model must be one of {', '.join(DIODE_MODELS)}")
        if x.type == "D" and tuple(x.nodes) == ("out", "0"):
            p.append(f'{x.id}: nodes ["out", "0"] put the anode on out, so the diode is forward-biased, holds out '
                     f'near 0.3-0.7 V and drains the storage. A zener clamp on out needs nodes ["0", "out"] '
                     f'(cathode on out)')
        if x.type == "S":
            if x.ctrl not in syncs:
                p.append(f"{x.id}: ctrl must name a SYNC component (got {x.ctrl!r})")
            else:
                used_syncs.add(x.ctrl)
        if x.type == "SYNC":
            if not _in(LIMITS["ton"], x.ton):
                p.append(f"{x.id}: ton {x.ton} outside [{LIMITS['ton'][0]:g}, {LIMITS['ton'][1]:g}] s")
            if not _in(LIMITS["phase"], 0.0 if x.phase is None else x.phase):
                p.append(f"{x.id}: phase {x.phase} outside [{LIMITS['phase'][0]}, {LIMITS['phase'][1]}] periods")
            if (x.edges or "both") not in EDGES:
                p.append(f"{x.id}: edges must be one of {', '.join(EDGES)}")
    for s in sorted(syncs - used_syncs):
        p.append(f"{s}: SYNC controller is not used by any switch")
    # connectivity
    count: Dict[str, int] = dict(HARNESS_CONNECTIONS)
    for x in c.components:
        if x.type in TWO_TERMINAL and len(x.nodes) == 2:
            for n in x.nodes:
                count[n] = count.get(n, 0) + 1
    for r in ("pzp", "pzn", "out"):
        if count.get(r, 0) <= HARNESS_CONNECTIONS[r]:
            p.append(f"node {r} is not connected to any component")
    for n, k in sorted(count.items()):
        if n not in RESERVED_NODES and k < 2:
            p.append(f"node {n} has only one connection (dangling)")
    if not _connected(c, "out", "pzp"):
        p.append("node out is not connected to the piezo (pzp) through the circuit")
    n_sw = sum(x.type == "S" for x in c.components)
    if len(c.components) > LIMITS["max_components"]:
        p.append(f"{len(c.components)} components; at most {LIMITS['max_components']} allowed")
    if n_sw > LIMITS["max_switches"]:
        p.append(f"{n_sw} switches; at most {LIMITS['max_switches']} allowed")
    if len(syncs) > LIMITS["max_sync"]:
        p.append(f"{len(syncs)} SYNC controllers; at most {LIMITS['max_sync']} allowed")
    return p


def _connected(c: Circuit, a: str, b: str) -> bool:
    adj: Dict[str, set] = {"pzp": {"pzn"}, "pzn": {"pzp"}}
    for x in c.components:
        if x.type in TWO_TERMINAL and len(x.nodes) == 2:
            u, v = x.nodes
            adj.setdefault(u, set()).add(v)
            adj.setdefault(v, set()).add(u)
    seen, stack = {a}, [a]
    while stack:
        for nb in adj.get(stack.pop(), ()):
            if nb not in seen:
                seen.add(nb)
                stack.append(nb)
    return b in seen


# ---------------------------------------------------------------- canonical form and hashes

def _h(s: str) -> str:
    return hashlib.sha1(s.encode()).hexdigest()[:16]


def _pins(x: Component) -> List[Tuple[str, str]]:
    """(node, role) pairs; diodes are polar (anode/cathode), others are not."""
    if x.type == "D":
        return [(x.nodes[0], "a"), (x.nodes[1], "k")]
    return [(n, "x") for n in x.nodes]


def canonical(c: Circuit) -> Circuit:
    """Same circuit with internal nodes renamed n1..nk and ids renamed by type, in a form that does
    not depend on the names or order the model chose."""
    comps = [copy.copy(x) for x in c.components]
    nodes = sorted({n for x in comps for n in x.nodes})
    label = {n: (n if n in RESERVED_NODES else "int") for n in nodes}
    for _ in range(3):
        new = {}
        for n in nodes:
            ctx = []
            for x in comps:
                pins = _pins(x) if x.type in TWO_TERMINAL else []
                for node, role in pins:
                    if node == n:
                        others = [label[o] for o, _r in pins if o != n]
                        ctx.append(f"{x.type}/{x.model}/{role}/{','.join(others)}")
            new[n] = _h(label[n] + "|" + ";".join(sorted(ctx)))
        label = {n: (n if n in RESERVED_NODES else new[n]) for n in nodes}

    by_id = {x.id: x for x in comps}

    def sig(x: Component) -> str:
        if x.type == "SYNC":
            sw = sorted(sig(s) for s in comps if s.type == "S" and s.ctrl == x.id)
            return f"SYNC/{x.edges or 'both'}/{'+'.join(sw)}"
        pins = _pins(x)
        labs = [f"{label[n]}{r}" for n, r in pins]
        if x.type != "D":
            labs.sort()
        extra = ""
        if x.type == "S" and x.ctrl in by_id:
            extra = "/" + (by_id[x.ctrl].edges or "both")
        return f"{x.type}/{x.model}/{','.join(labs)}{extra}"

    order = sorted(range(len(comps)), key=lambda i: (sig(comps[i]), i))
    rename: Dict[str, str] = {}
    k = 0
    for i in order:
        x = comps[i]
        for n in (x.nodes if x.type == "D" else sorted(x.nodes, key=lambda m: label[m])):
            if n not in RESERVED_NODES and n not in rename:
                k += 1
                rename[n] = f"n{k}"
    out: List[Component] = []
    counters: Dict[str, int] = {}
    id_map: Dict[str, str] = {}
    for i in order:
        x = comps[i]
        prefix = "G" if x.type == "SYNC" else x.type
        counters[prefix] = counters.get(prefix, 0) + 1
        id_map[x.id] = f"{prefix}{counters[prefix]}"
    for i in order:
        x = copy.copy(comps[i])
        nn = tuple(rename.get(n, n) for n in x.nodes)
        if x.type in TWO_TERMINAL and x.type != "D":
            nn = tuple(sorted(nn))
        x.nodes = nn
        x.id = id_map[x.id]
        if x.ctrl:
            x.ctrl = id_map.get(x.ctrl, x.ctrl)
        if x.type == "SYNC":
            x.edges = x.edges or "both"
            x.phase = 0.0 if x.phase is None else x.phase
        out.append(x)
    return Circuit(name=c.name, components=out, move=c.move, rationale=c.rationale, done=c.done)


def _struct_line(x: Component) -> str:
    return f"{x.id} {x.type} {' '.join(x.nodes)} m={x.model} ctrl={x.ctrl} e={x.edges}"


def _value_line(x: Component) -> str:
    vals = [f"{v:.3g}" if v is not None else "-" for v in (x.value, x.ton, x.phase)]
    return " ".join(vals)


def topology_hash(c: Circuit) -> str:
    cc = canonical(c)
    return hashlib.sha256("\n".join(_struct_line(x) for x in cc.components).encode()).hexdigest()[:12]


def design_hash(c: Circuit) -> str:
    cc = canonical(c)
    text = "\n".join(_struct_line(x) + " | " + _value_line(x) for x in cc.components)
    return hashlib.sha256(text.encode()).hexdigest()[:12]


# ---------------------------------------------------------------- serialisation and value edits

def to_dict(c: Circuit) -> dict:
    comps = []
    for x in c.components:
        d = {"id": x.id, "type": x.type}
        if x.type in TWO_TERMINAL:
            d["nodes"] = list(x.nodes)
        for f in ("value", "model", "ctrl", "ton", "phase", "edges"):
            v = getattr(x, f)
            if v is not None:
                d[f] = v
        comps.append(d)
    return {"move": c.move, "name": c.name, "rationale": c.rationale, "done": c.done, "components": comps}


def numeric_params(c: Circuit) -> List[Tuple[str, str]]:
    out = []
    for x in c.components:
        if x.type in VALUED:
            out.append((x.id, "value"))
        elif x.type == "SYNC":
            out += [(x.id, "ton"), (x.id, "phase")]
    return out


def _find(c: Circuit, cid: str) -> Component:
    for x in c.components:
        if x.id == cid:
            return x
    raise KeyError(cid)


def get_param(c: Circuit, cid: str, fld: str) -> float:
    v = getattr(_find(c, cid), fld)
    return 0.0 if v is None else float(v)


def with_param(c: Circuit, cid: str, fld: str, value: float) -> Circuit:
    """A copy of c with one numeric field changed, clamped to the allowed range."""
    new = copy.deepcopy(c)
    x = _find(new, cid)
    lo, hi = LIMITS[x.type] if fld == "value" else LIMITS[fld]
    setattr(x, fld, min(max(float(value), lo), hi))
    return new


def describe(c: Circuit) -> str:
    """One line per component, SPICE-like, for prompts and reports."""
    lines = []
    for x in c.components:
        if x.type == "SYNC":
            lines.append(f"{x.id} SYNC ton={x.ton:.3g}s phase={x.phase or 0:.3g} edges={x.edges or 'both'}")
        elif x.type == "S":
            lines.append(f"{x.id} S {x.nodes[0]} {x.nodes[1]} ctrl={x.ctrl}")
        elif x.type == "D":
            lines.append(f"{x.id} D {x.nodes[0]} {x.nodes[1]} {x.model}")
        else:
            lines.append(f"{x.id} {x.type} {x.nodes[0]} {x.nodes[1]} {x.value:.3g}")
    return "\n".join(lines)


In [ ]:
%%writefile piezo_synth/netlist.py
"""LTspice netlist for one circuit under one operating condition, plus the reference and witness circuits.

The harness owns everything except the proposed components: the piezo model, the load, controller
supply sinks, 1 GOhm bleed resistors on internal nodes and the piezo terminals (to avoid floating-node errors), device
models, the 0.8 s cold-start transient and the measurements.
"""
from __future__ import annotations

from typing import Tuple

from .circuit import Circuit, Component, canonical
from .spec import CTRL_SINK_A, LIMITS, SIM, Condition

MODELS_TEXT = """.model DSCH D(Is=5e-8 Rs=1 N=1.05 Cjo=10p BV=30 IBV=1u EG=0.69 XTI=2)
.model DSI D(Is=2.5n Rs=0.6 N=1.75 Cjo=4p BV=75 IBV=1u)
.model DZ33 D(Is=1e-12 Rs=5 N=1.5 BV=3.3 IBV=1u)
.model SWI SW(Ron=2 Roff=1G Vt=0.5 Vh=0)"""

MEAS_NAMES = ("vavg", "vavg_prev", "vpp", "vmax", "tup", "pload", "pext")


def fmt(v: float) -> str:
    return "%.6g" % v


def sync_timing(ton: float, phase: float, edges: str, freq: float) -> Tuple[float, float]:
    """(t0, period) of the controller's pulses. The piezo current is Ip*sin(2*pi*f*t), so the
    open-circuit piezo voltage peaks positive at t = 1/(2f) + k/f and negative at t = k/f.
    phase shifts the pulses by that fraction of the excitation period."""
    shift = (phase or 0.0) / freq
    if edges == "pos":
        period = 1.0 / freq
        t0 = (0.5 / freq + shift) % period
    elif edges == "neg":
        period = 1.0 / freq
        t0 = shift % period
    else:
        period = 0.5 / freq
        t0 = shift % period
    if abs(t0 - period) < 1e-15 or abs(t0) < 1e-15:
        t0 = 0.0
    return t0, period


def _component_line(x: Component) -> str:
    a, b = x.nodes if x.nodes else ("", "")
    if x.type in ("R", "C"):
        return f"{x.id} {a} {b} {fmt(x.value)}"
    if x.type == "L":
        rser = LIMITS["l_rser_base"] + LIMITS["l_rser_per_h"] * x.value
        return f"{x.id} {a} {b} {fmt(x.value)} Rser={fmt(rser)}"
    if x.type == "D":
        return f"{x.id} {a} {b} {x.model}"
    if x.type == "S":
        return f"{x.id} {a} {b} {x.ctrl}_g 0 SWI"
    raise ValueError(x.type)


def build_netlist(c: Circuit, cond: Condition, title: str = "", save_all: bool = False) -> str:
    cc = canonical(c)
    w0, w1 = SIM["win"]
    p0, p1 = SIM["win_prev"]
    tstop = SIM["tstop"]
    lines = [f"* {title or cc.name} | condition {cond.id}",
             f"Ipz pzn pzp SINE(0 {fmt(cond.ip)} {fmt(cond.freq)})",
             f"Cpz pzp pzn {fmt(cond.cp)}",
             f"Rpz pzp pzn {fmt(cond.rp)}",
             f"RL out 0 {fmt(cond.rl)}"]
    syncs = [x for x in cc.components if x.type == "SYNC"]
    for k, _ in enumerate(syncs, 1):
        lines.append(f"Ictl{k} out 0 {fmt(CTRL_SINK_A)}")
    for x in cc.components:
        if x.type != "SYNC":
            lines.append(_component_line(x))
    for x in syncs:
        t0, period = sync_timing(x.ton, x.phase, x.edges or "both", cond.freq)
        lines.append(f"V{x.id} {x.id}_g 0 PULSE(0 1 {fmt(t0)} 100n 100n {fmt(x.ton)} {fmt(period)})")
    # bleed resistors: internal nodes, and the piezo terminals (a design may couple them only through capacitors)
    internal = ["pzp", "pzn"] + sorted({n for x in cc.components for n in x.nodes} - {"0", "pzp", "pzn", "out"})
    for n in internal:
        lines.append(f"Rbleed_{n} {n} 0 1G")
    lines.append(MODELS_TEXT)
    lines.append(f".temp {fmt(cond.temp)}")
    lines.append(f".tran 0 {fmt(tstop)} 0 {fmt(SIM['maxstep'])}")
    if not save_all:
        lines.append(".save V(out) V(pzp) V(pzn) I(Ipz)")
    lines += [
        f".meas TRAN vavg AVG V(out) FROM={fmt(w0)} TO={fmt(w1)}",
        f".meas TRAN vavg_prev AVG V(out) FROM={fmt(p0)} TO={fmt(p1)}",
        f".meas TRAN vpp PP V(out) FROM={fmt(w0)} TO={fmt(w1)}",
        f".meas TRAN vmax MAX V(out) FROM=0 TO={fmt(tstop)}",
        ".meas TRAN tup FIND time WHEN V(out)=1.8 RISE=1",
        f".meas TRAN pload AVG V(out)*V(out)/{fmt(cond.rl)} FROM={fmt(w0)} TO={fmt(w1)}",
        f".meas TRAN pext AVG V(pzp,pzn)*I(Ipz) FROM={fmt(w0)} TO={fmt(w1)}",
        ".end",
    ]
    return "\n".join(lines) + "\n"


def _bridge() -> list:
    return [Component("D1", "D", ("pzp", "out"), model="DSCH"),
            Component("D2", "D", ("pzn", "out"), model="DSCH"),
            Component("D3", "D", ("0", "pzp"), model="DSCH"),
            Component("D4", "D", ("0", "pzn"), model="DSCH"),
            Component("C1", "C", ("out", "0"), value=2.2e-6)]


def reference_circuit() -> Circuit:
    """Full-bridge rectifier with a 2.2 uF storage capacitor: the standard passive interface."""
    return Circuit(name="FBR reference (4x Schottky, 2.2 uF)", components=_bridge(),
                   rationale="standard full-bridge rectifier")


def witness_circuit() -> Circuit:
    """Parallel SSHI + bridge + 3.3 V zener clamp. Used only by the smoke test to show the spec is
    reachable in principle; never shown to the model."""
    comps = _bridge() + [
        Component("L1", "L", ("pzp", "n1"), value=4.7e-3),
        Component("S1", "S", ("n1", "pzn"), ctrl="G1"),
        Component("G1", "SYNC", (), ton=34e-6, phase=0.0, edges="both"),
        Component("D5", "D", ("0", "out"), model="DZ33"),
    ]
    return Circuit(name="witness: p-SSHI + bridge + zener clamp", components=comps)


In [ ]:
%%writefile piezo_synth/sim.py
"""Run LTspice netlists in parallel threads, with a persistent content-hash cache and timing."""
from __future__ import annotations

import hashlib
import json
import threading
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable, Dict, List, Tuple


@dataclass
class SimResult:
    ok: bool
    meas: dict
    error: str
    seconds: float
    folder: str
    cached: bool = False


def ltspice_backend() -> Callable[[str, str, float], Any]:
    """The project's LTspice helper (in the ltspice-cpu image), imported on first use."""
    import sys
    if "/opt/ltspice" not in sys.path:
        sys.path.append("/opt/ltspice")
    from ltspice_run import simulate  # type: ignore

    def backend(text: str, folder: str, timeout: float):
        return simulate(text, folder, timeout=timeout)
    return backend


class Simulator:
    """backend(netlist_text, folder, timeout) -> object with .ok, .meas, .error (ltspice_run.simulate)."""

    def __init__(self, backend: Callable[[str, str, float], Any], root: Path, workers: int = 8,
                 timeout: float = 120.0):
        self.backend = backend
        self.root = Path(root)
        self.workers = max(1, int(workers))
        self.timeout = timeout
        self.root.mkdir(parents=True, exist_ok=True)
        self._cache_file = self.root / "sim_cache.jsonl"
        self._cache: Dict[str, SimResult] = {}
        self._lock = threading.Lock()
        self.stats = {"n_sims": 0, "cache_hits": 0, "sim_seconds": 0.0, "wall_seconds": 0.0, "failures": 0}
        if self._cache_file.exists():
            for line in self._cache_file.read_text(encoding="utf-8").splitlines():
                try:
                    d = json.loads(line)
                    self._cache[d["key"]] = SimResult(**d["result"])
                except (ValueError, KeyError, TypeError):
                    continue

    @staticmethod
    def key(text: str) -> str:
        return hashlib.sha256(text.encode()).hexdigest()

    def _one(self, key: str, text: str) -> SimResult:
        folder = self.root / "sims" / key[:16]
        t = time.time()
        try:
            r = self.backend(text, str(folder), self.timeout)
            res = SimResult(bool(getattr(r, "ok", False)), dict(getattr(r, "meas", {}) or {}),
                            str(getattr(r, "error", "") or ""), round(time.time() - t, 3), str(folder))
            cacheable = True
        except Exception as e:   # a crashed backend must not stop the search
            res = SimResult(False, {}, f"{type(e).__name__}: {e}\n{traceback.format_exc(limit=2)}",
                            round(time.time() - t, 3), str(folder))
            cacheable = False
        with self._lock:
            self.stats["n_sims"] += 1
            self.stats["sim_seconds"] += res.seconds
            self.stats["failures"] += 0 if res.ok else 1
            if cacheable:
                self._cache[key] = res
            if cacheable and res.ok:   # failures (timeouts, Wine trouble) are retried by a later run here
                with self._cache_file.open("a", encoding="utf-8") as f:
                    f.write(json.dumps({"key": key, "result": asdict(res)}) + "\n")
        return res

    def run(self, jobs: List[Tuple[str, str]]) -> List[SimResult]:
        """jobs: (label, netlist_text). Results come back in input order."""
        t = time.time()
        keys = [self.key(text) for _, text in jobs]
        todo: Dict[str, str] = {}
        with self._lock:
            for k, (_, text) in zip(keys, jobs):
                if k not in self._cache and k not in todo:
                    todo[k] = text
        fresh: Dict[str, SimResult] = {}
        if todo:
            with ThreadPoolExecutor(max_workers=min(self.workers, len(todo))) as ex:
                futs = {k: ex.submit(self._one, k, text) for k, text in todo.items()}
                for k, f in futs.items():
                    fresh[k] = f.result()
        out = []
        returned = set()
        with self._lock:
            for k in keys:
                if k in fresh and k not in returned:
                    returned.add(k)
                    out.append(fresh[k])
                else:
                    base = fresh.get(k) or self._cache[k]
                    if k not in fresh:
                        self.stats["cache_hits"] += 1
                    out.append(SimResult(base.ok, dict(base.meas), base.error, base.seconds, base.folder,
                                         cached=True))
            self.stats["wall_seconds"] += time.time() - t
        return out


In [ ]:
%%writefile piezo_synth/evaluate.py
"""Turn simulation measurements into pass/fail checks against the spec, and a scalar score."""
from __future__ import annotations

import math
from dataclasses import dataclass, field
from typing import Dict, List, Optional

from .spec import CRITERIA, Condition, Criterion


@dataclass
class Check:
    condition: str
    criterion: str
    measured: Optional[float]
    limit: float
    op: str
    margin: float
    passed: bool


@dataclass
class Evaluation:
    checks: List[Check] = field(default_factory=list)
    meas: Dict[str, dict] = field(default_factory=dict)
    sim_ok: Dict[str, bool] = field(default_factory=dict)
    sim_error: Dict[str, str] = field(default_factory=dict)
    scalar: float = 0.0
    cond_pass: int = 0
    checks_pass: int = 0
    all_pass: bool = False


def _num(v) -> Optional[float]:
    try:
        f = float(v)
    except (TypeError, ValueError):
        return None
    return f if math.isfinite(f) else None


def _clip(v: float, lo: float, hi: float) -> float:
    return min(max(v, lo), hi)


def evaluate(results: dict, conditions: List[Condition], criteria: List[Criterion] = CRITERIA) -> Evaluation:
    """results: condition id -> SimResult. Missing results count as failed simulations."""
    ev = Evaluation()
    for cond in conditions:
        r = results.get(cond.id)
        ok = bool(r is not None and r.ok)
        ev.sim_ok[cond.id] = ok
        if not ok:
            ev.sim_error[cond.id] = (r.error if r is not None else "not simulated") or "simulation failed"
        meas = {k: _num(v) for k, v in (r.meas.items() if ok else [])}
        if meas.get("vavg") is not None and meas.get("vavg_prev") is not None:
            meas["dv"] = abs(meas["vavg"] - meas["vavg_prev"])
        ev.meas[cond.id] = meas
        all_ok = True
        for crit in criteria:
            m = meas.get(crit.meas)
            if m is None:
                margin = -1.0
            elif crit.op == ">=":
                margin = (m - crit.limit) / crit.scale
            else:
                margin = (crit.limit - m) / crit.scale
            passed = m is not None and margin >= 0
            all_ok = all_ok and passed
            ev.checks.append(Check(cond.id, crit.id, m, crit.limit, crit.op, margin, passed))
        ev.cond_pass += int(all_ok)
    ev.checks_pass = sum(c.passed for c in ev.checks)
    ev.all_pass = bool(ev.checks) and ev.checks_pass == len(ev.checks)
    ev.scalar = (10 * ev.cond_pass + ev.checks_pass
                 + sum(_clip(c.margin, -1, 0) for c in ev.checks)
                 + 0.05 * sum(_clip(c.margin, 0, 0.5) for c in ev.checks))
    return ev


_UNITS = {c.id: c.unit for c in CRITERIA}


def diagnose(ev: Evaluation, limit: int = 8) -> str:
    """Failing checks, worst first, then simulation failures. Empty string if everything passed."""
    failed_sim = {k for k, ok in ev.sim_ok.items() if not ok}
    bad = sorted((c for c in ev.checks if not c.passed and c.condition not in failed_sim),
                 key=lambda c: c.margin)
    lines = []
    for c in bad[:limit]:
        u = _UNITS.get(c.criterion, "")
        got = "not reached / not measured" if c.measured is None else f"{c.measured:.4g} {u}"
        lines.append(f"{c.condition} {c.criterion}: {got} (needs {c.op} {c.limit:g} {u}, margin {c.margin:+.2f})")
    if len(bad) > limit:
        lines.append(f"... and {len(bad) - limit} more failing checks")
    for k in sorted(failed_sim):
        err = " ".join(ev.sim_error.get(k, "").split())[:200]
        lines.append(f"{k}: simulation failed: {err}")
    return "\n".join(lines)


def is_better(a: Evaluation, b: Optional[Evaluation], ncomp_a: int, ncomp_b: int) -> bool:
    """Higher scalar wins; on a tie, fewer components."""
    if b is None:
        return True
    if a.scalar > b.scalar + 1e-9:
        return True
    if abs(a.scalar - b.scalar) <= 1e-9:
        return ncomp_a < ncomp_b
    return False


In [ ]:
%%writefile piezo_synth/llm.py
"""Client for the cluster's Gemma 4 31B (vLLM, OpenAI-compatible API), and a scripted stand-in."""
from __future__ import annotations

import json
import re
import time
from dataclasses import dataclass
from typing import Callable, List, Optional

URL = "http://gemma-4-31b.cpd-instance.svc:3000/v1/chat/completions"
MODEL = "google/gemma-4-31b"


@dataclass
class LLMReply:
    answer: str = ""
    reasoning: str = ""
    seconds: float = 0.0
    prompt_tokens: Optional[int] = None
    completion_tokens: Optional[int] = None
    reasoning_tokens: Optional[int] = None
    finish: Optional[str] = None
    error: Optional[str] = None


def _requests_post(url, json, timeout):   # imported lazily: only needed for real calls
    import requests
    return requests.post(url, json=json, timeout=timeout)


class _HTTPError(Exception):
    def __init__(self, status: int, text: str):
        super().__init__(f"HTTP {status}: {text[:300]}")
        self.status = status


class GemmaClient:
    def __init__(self, url: str = URL, model: str = MODEL, timeout: float = 900, retries: int = 3,
                 backoff: float = 10.0, post: Callable = _requests_post, sleep: Callable = time.sleep):
        self.url, self.model, self.timeout = url, model, timeout
        self.retries, self.backoff = max(1, retries), backoff
        self.post, self.sleep = post, sleep

    def ask(self, messages: List[dict], think: bool, max_tokens: Optional[int] = None,
            temperature: Optional[float] = None, timeout: Optional[float] = None) -> LLMReply:
        """Never raises: on repeated failure the reply carries `error`. `timeout` (s) caps this call's
        read timeout, e.g. to the remaining wall-clock budget. A read timeout is not retried: resending
        the same request would take as long again."""
        read_timeout = self.timeout if timeout is None else max(1.0, min(self.timeout, timeout))
        body = {
            "model": self.model,
            "messages": messages,
            "max_tokens": max_tokens or (16384 if think else 2048),
            "temperature": temperature if temperature is not None else (1.0 if think else 0.3),
            "chat_template_kwargs": {"enable_thinking": bool(think)},
        }
        t = time.time()
        last = ""
        for attempt in range(1, self.retries + 1):
            try:
                r = self.post(self.url, json=body, timeout=read_timeout)
                status = getattr(r, "status_code", 200)
                if status >= 400:
                    text = getattr(r, "text", "") or ""
                    raise _HTTPError(status, text if isinstance(text, str) else "")
                out = r.json()
                choice = out["choices"][0]
                msg = choice.get("message") or {}
                usage = out.get("usage") or {}
                details = usage.get("completion_tokens_details") or {}
                return LLMReply(
                    answer=(msg.get("content") or "").strip(),
                    reasoning=msg.get("reasoning") or msg.get("reasoning_content") or "",
                    seconds=round(time.time() - t, 1),
                    prompt_tokens=usage.get("prompt_tokens"),
                    completion_tokens=usage.get("completion_tokens"),
                    reasoning_tokens=details.get("reasoning_tokens"),
                    finish=choice.get("finish_reason"),
                )
            except _HTTPError as e:
                last = str(e)
                if e.status < 500:
                    break
            except Exception as e:   # connection errors, timeouts, malformed bodies
                last = f"{type(e).__name__}: {e}"
                if "Timeout" in type(e).__name__:
                    break
            if attempt < self.retries:
                self.sleep(self.backoff * attempt)
        return LLMReply(seconds=round(time.time() - t, 1), error=last or "unknown error")


class ScriptedLLM:
    """Returns pre-written replies in order (repeating the last). For tests and --fake dry runs."""

    def __init__(self, replies: list):
        self.replies = list(replies)
        self.received: List[List[dict]] = []
        self._i = 0

    def ask(self, messages: List[dict], think: bool, max_tokens: Optional[int] = None,
            temperature: Optional[float] = None, timeout: Optional[float] = None) -> LLMReply:
        self.received.append(list(messages))
        r = self.replies[min(self._i, len(self.replies) - 1)]
        self._i += 1
        if isinstance(r, LLMReply):
            return r
        return LLMReply(answer=str(r), seconds=0.0, prompt_tokens=100, completion_tokens=50,
                        reasoning_tokens=None, finish="stop")


def json_from(text: str) -> Optional[dict]:
    """The circuit JSON in a reply: a fenced ```json block first, else the first object containing
    "components", else the first JSON object. None if there is none."""
    if not text:
        return None
    for m in re.finditer(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.S):
        try:
            d = json.loads(m.group(1))
            if isinstance(d, dict):
                return d
        except ValueError:
            pass
    dec = json.JSONDecoder()
    first = None
    for m in re.finditer(r"\{", text):
        try:
            d, _end = dec.raw_decode(text, m.start())
        except ValueError:
            continue
        if isinstance(d, dict):
            if "components" in d:
                return d
            if first is None:
                first = d
    return first


In [ ]:
%%writefile piezo_synth/prompts.py
"""Prompts for the model. Rebuilt from the search state every iteration (no growing chat history).

Only search conditions appear here: the held-out conditions are never shown to the model.
"""
from __future__ import annotations

import json
from typing import List, Optional

from . import spec
from .circuit import DIODE_MODELS, MOVES
from .evaluate import Evaluation

EXAMPLE_JSON = {
    "move": "new_topology",
    "name": "short label for this design",
    "rationale": "what you changed and why it should fix the failing checks",
    "done": False,
    "components": [
        {"id": "D1", "type": "D", "nodes": ["pzp", "out"], "model": "DSCH"},
        {"id": "C1", "type": "C", "nodes": ["out", "0"], "value": 2.2e-6},
        {"id": "L1", "type": "L", "nodes": ["pzp", "n1"], "value": 4.7e-3},
        {"id": "S1", "type": "S", "nodes": ["n1", "pzn"], "ctrl": "G1"},
        {"id": "G1", "type": "SYNC", "ton": 3.4e-5, "phase": 0.0, "edges": "both"},
    ],
}


def _f(v, fmt="{:.3g}") -> str:
    return "-" if v is None else fmt.format(v)


def condition_table(ev: Evaluation, conditions) -> str:
    """Fixed-width table: measurements and pass/fail per criterion for each condition."""
    crit = [c.id for c in spec.CRITERIA]
    head = f"{'cond':<5}{'vavg V':>8}{'vmax V':>8}{'vpp mV':>8}{'tup s':>8}{'Pload uW':>10}  " + \
        " ".join(f"{c:<8}" for c in crit)
    rows = [head]
    by = {(c.condition, c.criterion): c for c in ev.checks}
    for cond in conditions:
        m = ev.meas.get(cond.id, {})
        if not ev.sim_ok.get(cond.id, False):
            rows.append(f"{cond.id:<5}  simulation failed")
            continue
        vpp = m.get("vpp")
        pl = m.get("pload")
        flags = " ".join(f"{('ok' if by[(cond.id, c)].passed else 'FAIL'):<8}" for c in crit if (cond.id, c) in by)
        rows.append(f"{cond.id:<5}{_f(m.get('vavg')):>8}{_f(m.get('vmax')):>8}"
                    f"{_f(None if vpp is None else vpp * 1e3):>8}{_f(m.get('tup')):>8}"
                    f"{_f(None if pl is None else pl * 1e6):>10}  {flags}")
    return "\n".join(rows)


def system_prompt(reference_eval: Evaluation) -> str:
    conds = spec.search_conditions()
    ctable = "\n".join(
        f"  {c.id}: {c.label}; Ip = {c.ip * 1e6:g} uA peak, f = {c.freq:g} Hz, Cp = {c.cp * 1e9:g} nF, "
        f"RL = {c.rl / 1e3:g} kOhm, temperature {c.temp:g} C" for c in conds)
    crit = "\n".join(f"  {c.id}: {c.meas} {c.op} {c.limit:g} {c.unit} ({c.description})" for c in spec.CRITERIA)
    L = spec.LIMITS
    return f"""You are an expert power-electronics designer synthesising the interface circuit of a
piezoelectric energy harvester. You work by experiment: you propose a circuit, it is simulated in
LTspice under several operating conditions, and you receive the measurements.

GOAL
A piezo transducer must power a sensor-node load RL from a storage capacitor at node "out".
Under EVERY operating condition below, starting from 0 V (cold start) and simulating 0.8 s:
{crit}
Measurement windows: vavg and vpp over 0.7-0.8 s; vmax over the whole run; tup = first time
V(out) reaches 1.8 V (missing if never); dv = |mean(0.7-0.8 s) - mean(0.6-0.7 s)|.

PIEZO MODEL AND OPERATING CONDITIONS
The piezo is a sinusoidal current source Ip*sin(2*pi*f*t) in parallel with its capacitance Cp and a
20 MOhm leakage resistance, between nodes pzp and pzn (Cp = 25 nF nominal).
Its open-circuit voltage amplitude is Ip/(2*pi*f*Cp), about 3.2 V at nominal: low, so diode drops matter.
{ctable}

TEST BENCH (fixed, added automatically - do not include these)
- the piezo model between pzp and pzn; the load RL from out to 0;
- for every SYNC controller you use, a 0.5 uA supply current drawn from out (its power cost);
- a 1 GOhm resistor to 0 from pzp, pzn and every internal node (prevents floating nodes).

COMPONENTS YOU MAY USE (at most {L['max_components']} components)
- R  resistor, value {L['R'][0]:g}..{L['R'][1]:g} Ohm, nodes [a, b]
- C  capacitor, value {L['C'][0]:g}..{L['C'][1]:g} F, nodes [a, b]
- L  inductor, value {L['L'][0]:g}..{L['L'][1]:g} H; it has series resistance 2 Ohm + 5 Ohm per mH
- D  diode, nodes [anode, cathode]: forward current flows from the first node to the second.
     Model one of {', '.join(DIODE_MODELS)}:
     DSCH = low-drop Schottky (~0.25 V at uA), DSI = silicon small-signal (~0.5 V at uA),
     DZ33 = 3.3 V zener (reverse breakdown 3.3 V, conducts ~1 uA at 3.3 V and much more above)
     Orientation matters:
       full bridge: ["pzp", "out"], ["pzn", "out"], ["0", "pzp"], ["0", "pzn"] (return diodes have
       their anode on ground);
       zener clamp on out: ["0", "out"] (cathode on out, so it is reverse-biased and breaks down at
       3.3 V). ["out", "0"] would be forward-biased and hold out near 0.7 V; it is rejected.
- S  switch, nodes [a, b], "ctrl": id of a SYNC; Ron = 2 Ohm, Roff = 1 GOhm (at most {L['max_switches']})
- SYNC ideal synchronous controller (no nodes, at most {L['max_sync']}): closes its switches for "ton"
     seconds ({L['ton'][0]:g}..{L['ton'][1]:g}) at the piezo voltage extrema, which occur at the zero
     crossings of the piezo current. "edges": "both" (every half period), "pos" (positive peak only)
     or "neg" (negative peak only). "phase" shifts the pulses by that fraction of the excitation
     period ({L['phase'][0]}..{L['phase'][1]}). It tracks the excitation frequency automatically.
     For L-Cp inversion, ton is about pi*sqrt(L*Cp).
Nodes: "0" is ground; pzp, pzn are the piezo terminals; out is the load/storage node; name any other
nodes yourself (lower-case, e.g. n1, mid). Every node needs at least two connections, and out must
connect to the piezo through your circuit.

REFERENCE DESIGN: FBR (full-bridge rectifier, 4 x DSCH, 2.2 uF on out). Its results:
{condition_table(reference_eval, conds)}

HOW TO WORK
- Component values are tuned automatically by a numerical optimiser after each proposal, so spend
  your reasoning on circuit CONNECTIVITY (topology): which structure can meet every condition.
- Classify each proposal with "move": {', '.join(MOVES)} (resize = same connectivity, new values).
- Explain the expected effect in "rationale". Set "done": true only when you believe no further
  structural change can improve on the best design so far.
- Reply with ONE JSON object only, in this form:
{json.dumps(EXAMPLE_JSON, indent=1)}
"""


def required_move(moves: List[str]) -> Optional[str]:
    """Force a structural change after three value-only proposals in a row."""
    if len(moves) >= 3 and all(m == "resize" for m in moves[-3:]):
        return "new_topology or modify_topology"
    return None


def _design_block(d: dict) -> str:
    return (f"[{d['id']}] {d['name']}  (topology {d['topo_hash']}, score {d['scalar']:.2f}, "
            f"conditions passed {d['cond_pass']}/{len(spec.search_conditions())})\n"
            f"{d['circuit_text']}\n{d['table']}\n"
            f"failing: {d['diagnosis'] or 'nothing - all checks pass'}")


def iteration_prompt(iteration: int, max_proposals: int, top: List[dict], last: Optional[dict],
                     explored: List[dict], required: Optional[str]) -> str:
    parts = [f"Proposal {iteration} of {max_proposals}."]
    if last is None:
        parts.append("This is your first proposal: choose the topology you think most likely to meet "
                     "every condition.")
    elif last.get("invalid_problems"):
        parts.append("Your previous proposal was rejected before simulation:\n- " +
                     "\n- ".join(last["invalid_problems"]))
    elif last.get("duplicate"):
        parts.append(f"Your previous proposal '{last.get('name', '')}' was a duplicate of an earlier design "
                     f"({last.get('id', '?')}); nothing new was simulated. Propose something different.")
    elif last.get("error"):
        parts.append(f"Your previous request failed: {last['error']}")
    else:
        parts.append("Result of your previous proposal, with the values you proposed (it is tuned automatically "
                     "while you think; tuned versions appear under BEST DESIGNS):\n" + _design_block(last))
    if top:
        parts.append("BEST DESIGNS SO FAR (values already tuned):\n\n" + "\n\n".join(_design_block(d) for d in top))
    if explored:
        parts.append("Topologies explored so far (hash: name, best score):\n" + "\n".join(
            f"  {e['topo_hash']}: {e['name']} ({e['best_scalar']:.2f})" for e in explored))
    if required:
        parts.append(f"Your last three proposals only changed values. This proposal MUST change the "
                     f"connectivity (move = {required}).")
    parts.append("Reply with one JSON object only.")
    return "\n\n".join(parts)


def repair_messages(system: str, user: str, bad_answer: str, problems: List[str]) -> List[dict]:
    return [{"role": "system", "content": system},
            {"role": "user", "content": user},
            {"role": "assistant", "content": bad_answer[:4000]},
            {"role": "user", "content": "That reply could not be used:\n- " + "\n- ".join(problems) +
             "\nReply again with one corrected JSON object only."}]


In [ ]:
%%writefile piezo_synth/refine.py
"""Value tuning for a fixed topology: coordinate search, multiplicative for R/C/L/ton, additive for phase."""
from __future__ import annotations

import random
from typing import Callable, List, Tuple

from .circuit import Circuit, get_param, numeric_params, with_param
from .evaluate import Evaluation, is_better

MIN_FACTOR = 1.05
MIN_ADD = 0.005


def _initial_step(fld: str) -> float:
    if fld == "phase":
        return 0.05
    if fld == "ton":
        return 1.3
    return 2.0


def refine(circuit: Circuit, start: Evaluation, evaluate_fn: Callable[[Circuit], Evaluation], budget: int,
           rng: random.Random, should_stop: Callable[[], bool] = lambda: False
           ) -> Tuple[Circuit, Evaluation, List[Tuple[Circuit, Evaluation]]]:
    """Return (best circuit, its evaluation, every (circuit, evaluation) tried)."""
    params = numeric_params(circuit)
    tried: List[Tuple[Circuit, Evaluation]] = []
    if not params or budget <= 0:
        return circuit, start, tried
    ncomp = len(circuit.components)
    step = {p: _initial_step(p[1]) for p in params}
    best, best_ev = circuit, start
    seen = set()

    def small(p) -> bool:
        return step[p] < (MIN_ADD if p[1] == "phase" else MIN_FACTOR)

    while len(tried) < budget and not all(small(p) for p in params):
        improved = False
        order = list(params)
        rng.shuffle(order)
        for p in order:
            if small(p):
                continue
            cid, fld = p
            cur = get_param(best, cid, fld)
            if fld == "phase":
                cands = [cur + step[p], cur - step[p]]
            else:
                cands = [cur * step[p], cur / step[p]]
            for v in cands:
                if len(tried) >= budget or should_stop():
                    return best, best_ev, tried
                cand = with_param(best, cid, fld, v)
                newv = get_param(cand, cid, fld)
                key = (p, round(newv, 15), id(best))
                if abs(newv - cur) <= 1e-15 * max(1.0, abs(cur)) or key in seen:
                    continue   # clamped at a bound, or already tried from this point
                seen.add(key)
                ev = evaluate_fn(cand)
                tried.append((cand, ev))
                if is_better(ev, best_ev, ncomp, ncomp):
                    best, best_ev = cand, ev
                    improved = True
                    break
        if not improved:
            for p in params:
                step[p] = step[p] / 2 if p[1] == "phase" else step[p] ** 0.5
    return best, best_ev, tried


In [ ]:
%%writefile piezo_synth/records.py
"""Incremental run record: events, candidates, measurements, checks, progress, timing, model exchanges.

Every write is an append or an atomic replace, so a job that is killed leaves a usable record.
"""
from __future__ import annotations

import csv
import json
import os
import threading
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

from .circuit import Circuit, to_dict
from .evaluate import Evaluation
from .netlist import MEAS_NAMES

CANDIDATE_FIELDS = ["id", "iteration", "origin", "parent", "move", "name", "topo_hash", "design_hash",
                    "n_components", "split", "scalar", "cond_pass", "checks_pass", "all_pass"]
MEAS_FIELDS = ["candidate", "split", "condition", *MEAS_NAMES, "dv", "sim_ok", "sim_error"]
CHECK_FIELDS = ["candidate", "split", "condition", "criterion", "measured", "op", "limit", "margin", "passed"]


@dataclass
class Candidate:
    id: str
    iteration: int
    origin: str            # reference | witness | llm | refine
    parent: Optional[str]
    circuit: Circuit
    topo_hash: str
    design_hash: str
    evaluation: Evaluation
    move: str
    name: str
    n_components: int


class RunRecorder:
    def __init__(self, out_dir: Path, quiet: bool = False):
        self.out = Path(out_dir)
        self.out.mkdir(parents=True, exist_ok=True)
        self.quiet = quiet
        self.t0 = time.time()
        self._lock = threading.Lock()
        self._progress: dict = {}

    def elapsed(self) -> float:
        return time.time() - self.t0

    # -------------------------------------------------------------- console
    def log(self, phase: str, msg: str) -> None:
        if self.quiet:
            return
        e = int(self.elapsed())
        print(f"[{e // 3600:02d}:{e % 3600 // 60:02d}:{e % 60:02d}] [{phase:<7}] {msg}", flush=True)

    # -------------------------------------------------------------- files
    def write_json(self, name: str, obj) -> None:
        path = self.out / name
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_name(path.name + ".tmp")
        tmp.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
        os.replace(tmp, path)

    def _append_csv(self, name: str, fields, rows) -> None:
        path = self.out / name
        new = not path.exists()
        with path.open("a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
            if new:
                w.writeheader()
            for r in rows:
                w.writerow(r)

    def event(self, kind: str, **data) -> None:
        with self._lock:
            with (self.out / "events.jsonl").open("a", encoding="utf-8") as f:
                f.write(json.dumps({"t": round(self.elapsed(), 2), "kind": kind, **data}, default=str) + "\n")

    def candidate(self, c: Candidate, split: str = "search") -> None:
        ev = c.evaluation
        with self._lock:
            self._append_csv("candidates.csv", CANDIDATE_FIELDS, [{
                "id": c.id, "iteration": c.iteration, "origin": c.origin, "parent": c.parent or "",
                "move": c.move, "name": c.name, "topo_hash": c.topo_hash, "design_hash": c.design_hash,
                "n_components": c.n_components, "split": split, "scalar": round(ev.scalar, 4),
                "cond_pass": ev.cond_pass, "checks_pass": ev.checks_pass, "all_pass": ev.all_pass}])
            rows = []
            for cond, m in ev.meas.items():
                r = {"candidate": c.id, "split": split, "condition": cond, "sim_ok": ev.sim_ok.get(cond),
                     "sim_error": " ".join(ev.sim_error.get(cond, "").split())[:300]}
                r.update({k: m.get(k) for k in (*MEAS_NAMES, "dv")})
                rows.append(r)
            self._append_csv("measurements.csv", MEAS_FIELDS, rows)
            self._append_csv("checks.csv", CHECK_FIELDS, [
                {"candidate": c.id, "split": split, **{k: v for k, v in asdict(ch).items()}} for ch in ev.checks])
        folder = self.out / "candidates" / c.id
        folder.mkdir(parents=True, exist_ok=True)
        (folder / "circuit.json").write_text(json.dumps(to_dict(c.circuit), indent=1), encoding="utf-8")

    def progress(self, **fields) -> None:
        with self._lock:
            self._progress.update(fields)
            self._progress["elapsed_s"] = round(self.elapsed(), 1)
            snapshot = dict(self._progress)
        self.write_json("progress.json", snapshot)

    def llm_exchange(self, iteration: int, attempt: int, messages, reply) -> None:
        self.write_json(f"llm/iter-{iteration:02d}-{attempt}.json",
                        {"iteration": iteration, "attempt": attempt, "messages": messages, "reply": asdict(reply)})

    def timing(self, d: dict) -> None:
        self.write_json("timing.json", d)


In [ ]:
%%writefile piezo_synth/search.py
"""The search loop: the model proposes, LTspice evaluates, feedback goes back; value tuning runs while
the model thinks. Held-out conditions are evaluated only after the search has stopped."""
from __future__ import annotations

import json
import random
import shutil
import threading
import time
from dataclasses import asdict, dataclass
from typing import Dict, List, Optional, Set

from . import spec
from .circuit import Circuit, CircuitError, describe, design_hash, parse_circuit, to_dict, topology_hash
from .evaluate import Evaluation, diagnose, evaluate, is_better
from .llm import LLMReply, json_from
from .netlist import build_netlist, reference_circuit
from .prompts import condition_table, iteration_prompt, repair_messages, required_move, system_prompt
from .records import Candidate, RunRecorder
from .refine import refine
from .sim import Simulator

# a design whose output never reaches this on any condition is not value-tuned
DEAD_V = 0.5 * next(c.limit for c in spec.CRITERIA if c.id == "vout_min")


@dataclass
class SearchConfig:
    max_proposals: int = 16
    min_proposals: int = 4
    wall_clock_s: float = 3600
    stagnation: int = 6
    refine_budget: int = 10
    refine_min: int = 3          # tuning evaluations done even if the model replies quickly
    think: bool = True
    seed: int = 1
    continue_after_met: bool = False
    heartbeat_s: float = 15
    model_grace_s: float = 300   # a model call may run this long past wall_clock_s before it is abandoned


@dataclass
class SearchOutcome:
    selected: Candidate
    reference: Candidate
    stop_reason: str
    met_search: bool
    heldout_selected: Evaluation
    heldout_reference: Evaluation
    met_all: bool


def evaluate_circuit(c: Circuit, conditions, sim: Simulator, label: str) -> Evaluation:
    jobs = [(f"{label}/{cond.id}", build_netlist(c, cond, title=label)) for cond in conditions]
    results = sim.run(jobs)
    return evaluate({cond.id: r for cond, r in zip(conditions, results)}, conditions)


class _Search:
    def __init__(self, cfg: SearchConfig, llm, sim: Simulator, rec: RunRecorder):
        self.cfg, self.llm, self.sim, self.rec = cfg, llm, sim, rec
        self.conds = spec.search_conditions()
        self.rng = random.Random(cfg.seed)
        self.candidates: List[Candidate] = []
        self.by_design: Dict[str, Candidate] = {}
        self.refined: Set[str] = set()
        self.best: Optional[Candidate] = None
        self.reference: Optional[Candidate] = None
        self.latest_llm: Optional[Candidate] = None
        self.moves: List[str] = []
        self.t = {"model_s": 0.0, "model_calls": 0, "prompt_tokens": 0, "completion_tokens": 0,
                  "reasoning_tokens": 0, "refine_s": 0.0, "model_errors": 0}
        self.iteration = 0

    # ---------------------------------------------------------------- bookkeeping
    def _new_candidate(self, c: Circuit, ev: Evaluation, origin: str, parent: Optional[str]) -> Candidate:
        cand = Candidate(id=f"c{len(self.candidates):04d}", iteration=self.iteration, origin=origin,
                         parent=parent, circuit=c, topo_hash=topology_hash(c), design_hash=design_hash(c),
                         evaluation=ev, move=c.move if origin == "llm" else origin, name=c.name,
                         n_components=len(c.components))
        self.candidates.append(cand)
        self.by_design[cand.design_hash] = cand
        self.rec.candidate(cand)
        self.rec.event("candidate", id=cand.id, origin=origin, parent=parent, name=cand.name,
                       topo_hash=cand.topo_hash, scalar=round(ev.scalar, 4), cond_pass=ev.cond_pass,
                       checks_pass=ev.checks_pass, all_pass=ev.all_pass)
        if origin != "reference" and is_better(ev, self.best.evaluation if self.best else None,
                                               cand.n_components, self.best.n_components if self.best else 0):
            self.best = cand
            self.rec.log("eval", f"new best {cand.id} ({cand.name}): score {ev.scalar:.2f}, "
                                 f"{ev.cond_pass}/{len(self.conds)} conditions pass")
        elif origin == "reference" and self.best is None:
            self.best = cand
        self._progress()
        return cand

    def _progress(self, **extra) -> None:
        b = self.best
        self.rec.progress(iteration=self.iteration, max_proposals=self.cfg.max_proposals,
                          best_id=b.id if b else None, best_name=b.name if b else None,
                          best_scalar=round(b.evaluation.scalar, 3) if b else None,
                          best_cond_pass=b.evaluation.cond_pass if b else None,
                          n_candidates=len(self.candidates), **extra)

    def _evaluate(self, c: Circuit, label: str) -> Evaluation:
        self.rec.progress(phase="sim")
        self.rec.log("sim", f"{label}: simulating {len(self.conds)} conditions")
        t = time.time()
        ev = evaluate_circuit(c, self.conds, self.sim, label)
        self.rec.log("eval", f"{label}: score {ev.scalar:.2f}, {ev.cond_pass}/{len(self.conds)} conditions, "
                             f"{ev.checks_pass}/{len(ev.checks)} checks pass ({time.time() - t:.1f} s)")
        return ev

    def _account(self, reply: LLMReply) -> None:
        self.t["model_calls"] += 1
        self.t["model_s"] += reply.seconds or 0.0
        for k in ("prompt_tokens", "completion_tokens", "reasoning_tokens"):
            self.t[k] += getattr(reply, k) or 0
        if reply.error:
            self.t["model_errors"] += 1

    def _entry(self, cand: Candidate) -> dict:
        ev = cand.evaluation
        return {"id": cand.id, "name": cand.name, "topo_hash": cand.topo_hash, "scalar": ev.scalar,
                "cond_pass": ev.cond_pass, "circuit_text": describe(cand.circuit),
                "diagnosis": diagnose(ev), "table": condition_table(ev, self.conds)}

    def _top(self, k: int = 5) -> List[dict]:
        best_per_topo: Dict[str, Candidate] = {}
        for c in self.candidates:
            if c.origin == "reference":
                continue
            cur = best_per_topo.get(c.topo_hash)
            if cur is None or is_better(c.evaluation, cur.evaluation, c.n_components, cur.n_components):
                best_per_topo[c.topo_hash] = c
        ranked = sorted(best_per_topo.values(), key=lambda c: -c.evaluation.scalar)
        return [self._entry(c) for c in ranked[:k]]

    def _explored(self) -> List[dict]:
        out: Dict[str, dict] = {}
        for c in self.candidates:
            if c.origin == "reference":
                continue
            e = out.setdefault(c.topo_hash, {"topo_hash": c.topo_hash, "name": c.name, "best_scalar": c.evaluation.scalar})
            e["best_scalar"] = max(e["best_scalar"], c.evaluation.scalar)
        return list(out.values())

    # ---------------------------------------------------------------- refinement
    def _refine_target(self) -> Optional[Candidate]:
        for c in (self.latest_llm, self.best):
            if c is not None and c.origin != "reference" and c.id not in self.refined:
                if not self._alive(c):
                    self.refined.add(c.id)
                    self.rec.log("refine", f"not tuning {c.id} ({c.name}): output never reaches "
                                           f"{DEAD_V:g} V, so values alone are unlikely to fix it")
                    continue
                return c
        return None

    @staticmethod
    def _alive(c: Candidate) -> bool:
        """True if the output reaches half the minimum supply voltage on at least one condition."""
        return any((m.get("vavg") or 0.0) >= DEAD_V for m in c.evaluation.meas.values())

    def _refine(self, target: Candidate, reply_arrived) -> None:
        self.refined.add(target.id)
        self.rec.progress(phase="refine")
        self.rec.log("refine", f"tuning values of {target.id} ({target.name}), up to {self.cfg.refine_budget} evaluations")
        t = time.time()
        count = {"n": 0}

        def ev_fn(c: Circuit) -> Evaluation:
            count["n"] += 1
            dh = design_hash(c)
            if dh in self.by_design:
                return self.by_design[dh].evaluation
            ev = evaluate_circuit(c, self.conds, self.sim, f"{target.id}-tune")
            self._new_candidate(c, ev, "refine", target.id)
            return ev

        def stop() -> bool:
            return reply_arrived() and count["n"] >= self.cfg.refine_min

        best_c, best_ev, tried = refine(target.circuit, target.evaluation, ev_fn, self.cfg.refine_budget,
                                        self.rng, should_stop=stop)
        self.t["refine_s"] += time.time() - t
        self.rec.log("refine", f"{target.id}: {len(tried)} evaluations, score {target.evaluation.scalar:.2f} -> "
                               f"{best_ev.scalar:.2f} ({time.time() - t:.0f} s)")

    # ---------------------------------------------------------------- model call
    def _ask(self, messages: List[dict], think: bool, attempt: int) -> LLMReply:
        holder: dict = {}
        deadline = self.cfg.wall_clock_s + self.cfg.model_grace_s      # in run-elapsed seconds
        budget = max(0.0, deadline - self.rec.elapsed())

        def call():
            try:
                holder["r"] = self.llm.ask(messages, think=think, timeout=budget)
            except Exception as e:   # the client should not raise, but never let the thread die silently
                holder["r"] = LLMReply(error=f"{type(e).__name__}: {e}")

        th = threading.Thread(target=call, daemon=True)
        t0 = time.time()
        self.rec.progress(phase="model", model_wait_s=0)
        self.rec.log("model", f"proposal {self.iteration}: asking the model (thinking {'on' if think else 'off'})")
        th.start()
        if attempt == 1:
            target = self._refine_target()
            if target is not None and self.cfg.refine_budget > 0:
                self._refine(target, lambda: not th.is_alive() or self.rec.elapsed() >= deadline)
        last_beat = time.time()
        while th.is_alive():
            if self.rec.elapsed() >= deadline:
                # the HTTP call is still hanging: abandon the (daemon) thread rather than overrun the budget
                holder.setdefault("r", LLMReply(seconds=round(time.time() - t0, 1),
                                                error="model call abandoned: wall-clock budget exceeded"))
                break
            th.join(timeout=min(1.0, self.cfg.heartbeat_s))
            if th.is_alive() and time.time() - last_beat >= self.cfg.heartbeat_s:
                last_beat = time.time()
                waited = time.time() - t0
                self.rec.progress(phase="model", model_wait_s=round(waited))
                self.rec.log("model", f"waiting {waited:.0f} s (proposal {self.iteration})")
        reply = holder.get("r") or LLMReply(error="model call abandoned: wall-clock budget exceeded")
        self._account(reply)
        self.rec.llm_exchange(self.iteration, attempt, messages, reply)
        tok = f"{reply.prompt_tokens} in / {reply.completion_tokens} out" if reply.completion_tokens is not None else "no token counts"
        self.rec.log("model", f"reply after {reply.seconds:.0f} s ({tok})" + (f"; ERROR {reply.error}" if reply.error else ""))
        self.rec.event("model_reply", iteration=self.iteration, attempt=attempt, seconds=reply.seconds,
                       prompt_tokens=reply.prompt_tokens, completion_tokens=reply.completion_tokens,
                       reasoning_tokens=reply.reasoning_tokens, finish=reply.finish, error=reply.error)
        return reply

    @staticmethod
    def _parse(answer: str):
        obj = json_from(answer)
        if obj is None:
            return None, ["no JSON object found in the reply"]
        try:
            return parse_circuit(obj), []
        except CircuitError as e:
            return None, e.problems

    # ---------------------------------------------------------------- main loop
    def run(self) -> SearchOutcome:
        cfg, rec = self.cfg, self.rec
        rec.write_json("spec.json", spec.spec_dict())
        rec.write_json("config.json", asdict(cfg))
        rec.progress(phase="setup", stop_reason=None)
        rec.log("setup", f"reference circuit on {len(self.conds)} search conditions")
        ref_c = reference_circuit()
        ref_c.move = "new_topology"
        self.reference = self._new_candidate(ref_c, self._evaluate(ref_c, "reference"), "reference", None)
        system = system_prompt(self.reference.evaluation)
        last: Optional[dict] = None
        consecutive_errors = no_improve = 0
        stop_reason = "proposal_budget"
        for i in range(1, cfg.max_proposals + 1):
            self.iteration = i
            if rec.elapsed() >= cfg.wall_clock_s:
                stop_reason = "time_budget"
                break
            before = self.best.evaluation.scalar
            user = iteration_prompt(i, cfg.max_proposals, self._top(), last, self._explored(), required_move(self.moves))
            messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
            reply = self._ask(messages, cfg.think, 1)
            done = False
            if reply.error:
                consecutive_errors += 1
                last = {"error": reply.error}
                rec.event("model_error", iteration=i, error=reply.error)
            else:
                consecutive_errors = 0
                circuit, problems = self._parse(reply.answer)
                if problems:
                    rec.log("eval", f"proposal {i} rejected ({len(problems)} problems); asking for a correction")
                    rec.event("proposal_rejected", iteration=i, problems=problems)
                    reply2 = self._ask(repair_messages(system, user, reply.answer, problems), False, 2)
                    if not reply2.error:
                        circuit, problems = self._parse(reply2.answer)
                if circuit is None:
                    rec.event("proposal_invalid", iteration=i, problems=problems)
                    rec.log("eval", f"proposal {i} unusable: {'; '.join(problems)[:200]}")
                    last = {"invalid_problems": problems}
                else:
                    done = circuit.done
                    self.moves.append(circuit.move)
                    dh = design_hash(circuit)
                    if dh in self.by_design:
                        prev = self.by_design[dh]
                        rec.event("duplicate", iteration=i, of=prev.id, name=circuit.name)
                        rec.log("eval", f"proposal {i} duplicates {prev.id}; not simulated")
                        last = {"duplicate": True, "id": prev.id, "name": circuit.name}
                    else:
                        rec.event("proposal", iteration=i, move=circuit.move, name=circuit.name,
                                  rationale=circuit.rationale, circuit=to_dict(circuit))
                        rec.log("eval", f"proposal {i}: {circuit.move} '{circuit.name}' ({len(circuit.components)} parts)")
                        cand = self._new_candidate(circuit, self._evaluate(circuit, f"proposal-{i:02d}"), "llm", None)
                        self.latest_llm = cand
                        last = dict(self._entry(cand), duplicate=False, invalid_problems=[], refined_from=None)
            no_improve = 0 if self.best.evaluation.scalar >= before + 0.01 else no_improve + 1
            self._progress(phase="eval")
            if self.best.evaluation.all_pass and not cfg.continue_after_met:
                stop_reason = "search_spec_met"
                break
            if done and i >= cfg.min_proposals:
                stop_reason = "model_done"
                break
            if consecutive_errors >= 3:
                stop_reason = "model_unavailable"
                break
            if no_improve >= cfg.stagnation:
                stop_reason = "stagnation"
                break
            if rec.elapsed() >= cfg.wall_clock_s:
                stop_reason = "time_budget"
                break
        return self._finish(stop_reason)

    def _finish(self, stop_reason: str) -> SearchOutcome:
        rec = self.rec
        sel, ref = self.best, self.reference
        rec.log("done", f"search stopped: {stop_reason}; selected {sel.id} ({sel.name})")
        rec.event("stop", reason=stop_reason, selected=sel.id)
        held = spec.heldout_conditions()
        rec.progress(phase="heldout", stop_reason=stop_reason)
        rec.log("heldout", f"checking {sel.id} and the reference on {len(held)} held-out conditions")
        ev_sel = evaluate_circuit(sel.circuit, held, self.sim, f"{sel.id}-heldout")
        ev_ref = ev_sel if sel is ref else evaluate_circuit(ref.circuit, held, self.sim, "reference-heldout")
        pairs = [(sel, ev_sel)] if sel is ref else [(sel, ev_sel), (ref, ev_ref)]
        for cand, ev in pairs:
            rec.candidate(Candidate(**{**cand.__dict__, "evaluation": ev}), split="heldout")
        met_search = sel.evaluation.all_pass
        met_all = met_search and ev_sel.all_pass
        rec.log("heldout", f"{sel.id}: {ev_sel.cond_pass}/{len(held)} held-out conditions pass; "
                           f"reference: {ev_ref.cond_pass}/{len(held)}")
        # selected design: circuit and the exact netlists for every condition
        sdir = rec.out / "selected"
        if sdir.exists():
            shutil.rmtree(sdir)
        sdir.mkdir(parents=True)
        (sdir / "circuit.json").write_text(json.dumps(to_dict(sel.circuit), indent=1), encoding="utf-8")
        for cond in spec.CONDITIONS:
            (sdir / f"{cond.id}.net").write_text(build_netlist(sel.circuit, cond, title=sel.id), encoding="utf-8")
        st = self.sim.stats
        timing = {"wall_s": round(rec.elapsed(), 1), "model_s": round(self.t["model_s"], 1),
                  "model_calls": self.t["model_calls"], "model_errors": self.t["model_errors"],
                  "prompt_tokens": self.t["prompt_tokens"], "completion_tokens": self.t["completion_tokens"],
                  "reasoning_tokens": self.t["reasoning_tokens"],
                  "sim_seconds": round(st["sim_seconds"], 1), "sim_wall_s": round(st["wall_seconds"], 1),
                  "n_sims": st["n_sims"], "cache_hits": st["cache_hits"], "sim_failures": st.get("failures", 0),
                  "refine_s": round(self.t["refine_s"], 1),
                  "note": "model calls overlap value tuning and simulations run in parallel, so the sums can exceed wall_s"}
        rec.timing(timing)
        failed = [{"split": split, **c.__dict__} for split, ev in (("search", sel.evaluation), ("heldout", ev_sel))
                  for c in ev.checks if not c.passed]
        rec.write_json("outcome.json", {"stop_reason": stop_reason, "met_search": met_search, "met_all": met_all,
                                        "selected_id": sel.id, "selected_name": sel.name,
                                        "selected_origin": sel.origin, "reference_id": ref.id,
                                        "failed_checks": failed})
        rec.progress(phase="done", stop_reason=stop_reason)
        rec.log("done", f"spec met on search conditions: {met_search}; on all conditions: {met_all}")
        return SearchOutcome(sel, ref, stop_reason, met_search, ev_sel, ev_ref, met_all)


def run_search(cfg: SearchConfig, llm, sim: Simulator, rec: RunRecorder) -> SearchOutcome:
    return _Search(cfg, llm, sim, rec).run()


In [ ]:
%%writefile piezo_synth/report.py
"""Build report.md and summary.json from a run folder (works on incomplete runs too).

    python -m piezo_synth.report runs/<run_id>
"""
from __future__ import annotations

import csv
import json
import sys
from collections import defaultdict
from pathlib import Path
from typing import Dict, List

from . import spec as spec_mod
from .circuit import describe, parse_circuit


def _read_csv(path: Path, required=()) -> List[dict]:
    """Rows of a CSV; rows cut short by a killed or still-writing job (missing required fields) are skipped."""
    if not path.exists():
        return []
    with path.open(newline="", encoding="utf-8") as f:
        return [r for r in csv.DictReader(f) if all(r.get(k) not in (None, "") for k in required)]


def _read_json(path: Path, default=None):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return default


def _events(path: Path) -> List[dict]:
    out = []
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            try:
                out.append(json.loads(line))
            except ValueError:
                continue   # a line cut off by a killed job
    return out


def _f(v, fmt="{:.3g}", scale=1.0) -> str:
    if v in (None, ""):
        return "-"
    try:
        return fmt.format(float(v) * scale)
    except ValueError:
        return str(v)


def _table(head: List[str], rows: List[List[str]]) -> str:
    lines = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
    lines += ["| " + " | ".join(str(c).replace("|", "/") for c in r) + " |" for r in rows]
    return "\n".join(lines)


def _true(v) -> bool:
    return str(v).lower() in ("true", "1")


def build_report(out_dir) -> str:
    out = Path(out_dir)
    spec = _read_json(out / "spec.json") or spec_mod.spec_dict()
    cands = _read_csv(out / "candidates.csv", ("id", "split", "scalar", "n_components", "all_pass"))
    meas = _read_csv(out / "measurements.csv", ("candidate", "condition"))
    checks = _read_csv(out / "checks.csv", ("candidate", "condition", "criterion", "limit", "margin", "passed"))
    events = _events(out / "events.jsonl")
    timing = _read_json(out / "timing.json", {}) or {}
    outcome = _read_json(out / "outcome.json")
    progress = _read_json(out / "progress.json", {}) or {}
    crit_ids = [c["id"] for c in spec["criteria"]]
    conds = spec["conditions"]
    search_ids = [c["id"] for c in conds if c["split"] == "search"]
    held_ids = [c["id"] for c in conds if c["split"] == "heldout"]

    complete = outcome is not None and progress.get("phase") == "done"
    search_rows = [c for c in cands if c["split"] == "search"]
    if outcome:
        sel_id, ref_id = outcome["selected_id"], outcome["reference_id"]
        stop_reason = outcome["stop_reason"] if complete else "incomplete"
    else:
        stop_reason = "incomplete"
        ref_id = next((c["id"] for c in search_rows if c["origin"] == "reference"), None)
        others = [c for c in search_rows if c["origin"] != "reference"] or search_rows
        best = max(others, key=lambda c: (float(c["scalar"]), -int(c["n_components"])), default=None)
        sel_id = best["id"] if best else None
    by_id = {c["id"]: c for c in search_rows}
    sel = by_id.get(sel_id, {})

    M: Dict[tuple, dict] = {(m["candidate"], m["condition"]): m for m in meas}
    K: Dict[tuple, Dict[str, dict]] = defaultdict(dict)
    for k in checks:
        K[(k["candidate"], k["condition"])][k["criterion"]] = k

    def cond_rows(cid: str, ids: List[str]) -> List[List[str]]:
        rows = []
        for cond in ids:
            m = M.get((cid, cond))
            if m is None:
                continue
            ks = K.get((cid, cond), {})
            flags = ["ok" if _true(ks[c]["passed"]) else "**FAIL**" if c in ks else "-" for c in crit_ids]
            if not _true(m.get("sim_ok")):
                rows.append([cond, "sim failed", "", "", "", "", ""] + flags)
                continue
            rows.append([cond, _f(m["vavg"]), _f(m["vmax"]), _f(m["vpp"], scale=1e3), _f(m["tup"]),
                         _f(m["pload"], scale=1e6), _f(m["pext"], scale=1e6)] + flags)
        return rows

    mhead = ["cond", "Vavg V", "Vmax V", "ripple mV", "t_up s", "P_load uW", "P_ext uW"] + crit_ids

    def failed(cid: str, ids: List[str]) -> List[dict]:
        res = []
        for cond in ids:
            for c in crit_ids:
                k = K.get((cid, cond), {}).get(c)
                if k is not None and not _true(k["passed"]):
                    res.append({"condition": cond, "criterion": c, "measured": k["measured"] or None,
                                "op": k["op"], "limit": float(k["limit"]), "margin": float(k["margin"])})
        return res

    def all_pass(cid: str, ids: List[str]) -> bool:
        have = [K.get((cid, c)) for c in ids]
        return all(have) and not failed(cid, ids)

    s = ["# Piezoelectric harvester topology synthesis: run report", ""]
    s.append(f"Run folder: `{out}`. Stop reason: **{stop_reason}**. Selected design: **{sel_id}** "
             f"({sel.get('name', '?')}).")
    # 1. specification
    s += ["", "## Specification", "",
          "Every criterion must hold under every operating condition (0.8 s cold-start transient).", ""]
    s.append(_table(["criterion", "measurement", "requirement", "meaning"],
                    [[c["id"], c["meas"], f"{c['op']} {c['limit']:g} {c['unit']}", c["description"]]
                     for c in spec["criteria"]]))
    lim = spec["limits"]
    s += ["", f"Limits: at most {lim['max_components']} components, {lim['max_switches']} switches, "
              f"{lim['max_sync']} SYNC controllers (each draws {spec.get('ctrl_sink_a', 0.5e-6) * 1e6:g} uA from out); "
              f"L {lim['L'][0]:g}-{lim['L'][1]:g} H with series resistance {lim['l_rser_base']:g} Ohm + "
              f"{lim['l_rser_per_h'] / 1000:g} Ohm/mH; C up to {lim['C'][1]:g} F."]
    # 2. conditions
    s += ["", "## Test conditions", ""]
    s.append(_table(["id", "set", "description", "Ip uA pk", "f Hz", "Cp nF", "RL kOhm", "T C"],
                    [[c["id"], "search" if c["split"] == "search" else "held-out", c["label"], _f(c["ip"], scale=1e6),
                      _f(c["freq"]), _f(c["cp"], scale=1e9), _f(c["rl"], scale=1e-3), _f(c["temp"])] for c in conds]))
    s += ["", "Held-out conditions were never shown to the model and were not used for scoring, selection or stopping."]
    # 3. reference
    s += ["", "## Reference results", "",
          f"Reference: full-bridge rectifier, 4 x Schottky, 2.2 uF (candidate {ref_id})."]
    s += ["", _table(mhead, cond_rows(ref_id, search_ids + held_ids))] if ref_id else ["", "(not evaluated)"]
    # 4. proposals
    s += ["", "## Proposals tried", ""]
    rows = []
    per_iter: Dict[int, dict] = defaultdict(lambda: {"s": 0.0, "tok": 0, "calls": 0})
    for e in events:
        if e["kind"] == "model_reply":
            d = per_iter[e["iteration"]]
            d["s"] += e.get("seconds") or 0
            d["tok"] += (e.get("completion_tokens") or 0)
            d["calls"] += 1
    llm_by_iter = {int(c["iteration"]): c for c in search_rows if c["origin"] == "llm"}
    outcome_by_iter: Dict[int, str] = {}
    for e in events:
        it = e.get("iteration")
        if e["kind"] == "duplicate":
            outcome_by_iter[it] = f"duplicate of {e['of']}"
        elif e["kind"] == "proposal_invalid":
            outcome_by_iter[it] = "invalid: " + "; ".join(e.get("problems", []))[:120]
        elif e["kind"] == "model_error":
            outcome_by_iter[it] = "model error: " + str(e.get("error"))[:80]
    for it in sorted(set(per_iter) | set(llm_by_iter) | set(outcome_by_iter)):
        c = llm_by_iter.get(it)
        d = per_iter.get(it, {"s": 0, "tok": 0, "calls": 0})
        if c:
            rows.append([it, c["id"], c["move"], c["name"], c["topo_hash"], "simulated",
                         f"{c['cond_pass']}/{len(search_ids)}", _f(c["scalar"], "{:.2f}"), f"{d['s']:.0f}", d["tok"] or "-"])
        else:
            rows.append([it, "-", "-", "-", "-", outcome_by_iter.get(it, "-"), "-", "-", f"{d['s']:.0f}", d["tok"] or "-"])
    s.append(_table(["#", "candidate", "move", "name", "topology", "outcome", "cond. pass", "score",
                     "model s", "completion tokens"], rows) if rows else "(none)")
    refines = [c for c in search_rows if c["origin"] == "refine"]
    if refines:
        grp: Dict[str, List[dict]] = defaultdict(list)
        for c in refines:
            grp[c["parent"]].append(c)
        s += ["", "Value tuning (automatic, between model calls):", ""]
        s.append(_table(["tuned from", "evaluations", "start score", "best score", "best candidate"],
                        [[p, len(v), _f(by_id.get(p, {}).get("scalar"), "{:.2f}"),
                          _f(max(float(x["scalar"]) for x in v), "{:.2f}"),
                          max(v, key=lambda x: float(x["scalar"]))["id"]] for p, v in grp.items()]))
    ntopo = len({c["topo_hash"] for c in search_rows if c["origin"] != "reference"})
    s += ["", f"{len(search_rows)} designs evaluated on the search conditions, {ntopo} distinct topologies "
              f"(excluding the reference)."]
    # 5. selected design
    s += ["", "## Selected design", ""]
    circ = _read_json(out / "candidates" / str(sel_id) / "circuit.json") if sel_id else None
    if circ:
        try:
            text = describe(parse_circuit(circ))
        except Exception:   # an old or hand-edited file: show it raw
            text = json.dumps(circ, indent=1)
        s += [f"**{sel_id}**: {sel.get('name', '')} (origin {sel.get('origin', '?')}, topology {sel.get('topo_hash', '?')})",
              "", "```", text, "```", ""]
    s.append(_table(mhead, cond_rows(sel_id, search_ids)) if sel_id else "(none)")
    # 6. held-out
    s += ["", "## Held-out check", ""]
    if any((sel_id, c) in M for c in held_ids):
        s.append("Selected design:")
        s += ["", _table(mhead, cond_rows(sel_id, held_ids)), ""]
        if ref_id and ref_id != sel_id:
            s += ["Reference:", "", _table(mhead, cond_rows(ref_id, held_ids))]
    else:
        s.append("Not run (the search did not finish).")
    # 7. unmet
    fails = [dict(f, split="search") for f in failed(sel_id, search_ids)] + \
            [dict(f, split="heldout") for f in failed(sel_id, held_ids)] if sel_id else []
    s += ["", "## Unmet requirements", ""]
    if fails:
        s.append(_table(["set", "condition", "criterion", "measured", "requirement", "margin"],
                        [[f["split"], f["condition"], f["criterion"],
                          _f(f["measured"]) if f["measured"] else "not reached", f"{f['op']} {f['limit']:g}",
                          f"{f['margin']:+.2f}"] for f in fails]))
    else:
        s.append("None: the selected design meets every criterion on every evaluated condition.")
    met_search = bool(sel_id) and all_pass(sel_id, search_ids)
    met_all = met_search and all_pass(sel_id, held_ids)
    # 8. stop reason
    explain = {
        "search_spec_met": "a design met every criterion on every search condition",
        "model_done": "the model declared that no further structural change would help",
        "model_unavailable": "three model calls in a row failed",
        "stagnation": "the best score stopped improving",
        "time_budget": "the wall-clock budget was used up",
        "proposal_budget": "the maximum number of proposals was reached",
        "smoke": "smoke test (reference and witness only, no model)",
        "incomplete": "the job ended before the search finished (killed, timed out, or still running)",
    }
    s += ["", "## Stop reason", "", f"**{stop_reason}**: {explain.get(stop_reason, '')}.",
          f"Spec met on search conditions: **{met_search}**; on all conditions including held-out: **{met_all}**."]
    # 9. time
    s += ["", "## Time and tokens", ""]
    if timing:
        s.append(_table(["quantity", "value"], [
            ["wall time", f"{timing.get('wall_s', 0):.0f} s"],
            ["model calls", timing.get("model_calls")], ["time in model calls", f"{timing.get('model_s', 0):.0f} s"],
            ["prompt tokens", timing.get("prompt_tokens")], ["completion tokens", timing.get("completion_tokens")],
            ["of which reasoning tokens", timing.get("reasoning_tokens") or "not reported"],
            ["simulations run", timing.get("n_sims")], ["cache hits (not re-simulated)", timing.get("cache_hits")],
            ["simulator time (sum over parallel runs)", f"{timing.get('sim_seconds', 0):.0f} s"],
            ["simulation wall time", f"{timing.get('sim_wall_s', 0):.0f} s"],
            ["value tuning wall time", f"{timing.get('refine_s', 0):.0f} s"]]))
        s += ["", "Value tuning runs while the model is thinking, so model and simulation times overlap."]
    else:
        s.append("Not available (run incomplete).")
    # 10. files
    s += ["", "## Files", "", _table(["file", "contents"], [
        ["selected/circuit.json, selected/<cond>.net", "selected design and its exact netlist for every condition"],
        ["candidates.csv", "every design evaluated (one row per design and condition set)"],
        ["measurements.csv", "every measurement, per design and condition"],
        ["checks.csv", "every pass/fail decision with margin"],
        ["candidates/<id>/circuit.json", "each design's circuit"],
        ["llm/iter-NN-a.json", "prompts, replies and model reasoning"],
        ["events.jsonl", "timeline of the run"], ["sims/<hash>/", "LTspice netlists, logs and waveforms"],
        ["timing.json, outcome.json, spec.json, config.json", "timing, result, spec and settings"]])]
    text = "\n".join(s) + "\n"
    (out / "report.md").write_text(text, encoding="utf-8")
    summary = {"stop_reason": stop_reason, "met_search": met_search, "met_all": met_all, "selected_id": sel_id,
               "selected_topology": {"name": sel.get("name"), "topo_hash": sel.get("topo_hash")},
               "reference_id": ref_id, "failed_checks": fails, "timing": timing}
    (out / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return text


if __name__ == "__main__":
    print(build_report(sys.argv[1] if len(sys.argv) > 1 else "."))


In [ ]:
%%writefile piezo_synth/rawread.py
"""Minimal reader for LTspice .raw waveform files (real-valued transient data, binary or ASCII).

Use uncompressed files (`.options plotwinsize=0`, as reproduce.py --waveforms does).
"""
from __future__ import annotations

import struct
from pathlib import Path
from typing import Dict, List


def _decode_header(raw: bytes):
    """Return (header_text, data_start, kind) with kind 'binary' or 'ascii'."""
    utf16 = len(raw) > 1 and raw[1] == 0
    enc = "utf-16-le" if utf16 else "latin-1"
    for marker, kind in (("Binary:\n", "binary"), ("Values:\n", "ascii")):
        m = marker.encode(enc)
        i = raw.find(m)
        if i < 0 and not utf16:
            m = marker.replace("\n", "\r\n").encode(enc)
            i = raw.find(m)
        if i >= 0:
            return raw[:i].decode(enc), i + len(m), kind, enc
    raise ValueError("not an LTspice raw file: no 'Binary:' or 'Values:' section")


def read_raw(path) -> Dict[str, List[float]]:
    raw = Path(path).read_bytes()
    header, start, kind, enc = _decode_header(raw)
    lines = header.replace("\r\n", "\n").split("\n")
    info, names = {}, []
    in_vars = False
    for line in lines:
        if in_vars:
            parts = line.split()
            if len(parts) >= 3:
                names.append(parts[1])
            continue
        if line.startswith("Variables:"):
            in_vars = True
            continue
        if ":" in line:
            k, v = line.split(":", 1)
            info[k.strip().lower()] = v.strip()
    nvars = int(info["no. variables"])
    npts = int(info["no. points"])
    names = names[:nvars]
    flags = info.get("flags", "").lower()
    if "complex" in flags:
        raise ValueError("complex (AC) data is not supported")
    out: Dict[str, List[float]] = {n: [] for n in names}
    if kind == "binary":
        double = "double" in flags
        fmt = "<" + "d" * nvars if double else "<d" + "f" * (nvars - 1)
        size = struct.calcsize(fmt)
        data = raw[start:]
        npts = min(npts, len(data) // size)
        for k in range(npts):
            vals = struct.unpack_from(fmt, data, k * size)
            for n, v in zip(names, vals):
                out[n].append(v)
    else:
        toks = raw[start:].decode(enc).split()
        step = nvars + 1
        for k in range(min(npts, len(toks) // step)):
            vals = toks[k * step + 1:(k + 1) * step]
            for n, v in zip(names, vals):
                out[n].append(float(v.split(",")[0]))
    if names and names[0].lower() == "time":
        out[names[0]] = [abs(t) for t in out[names[0]]]   # LTspice marks some points with a negative time
    return out


In [ ]:
%%writefile piezo_synth/run.py
"""Entry point for the cluster job.

    python3 -u -m piezo_synth.run --config run_config.json --out runs/<id>    full experiment
    python3 -u -m piezo_synth.run --smoke --out runs/smoke                   reference + witness, no model
    python3 -u -m piezo_synth.run --fake --out runs/dry                      wiring check without LTspice/Gemma
"""
from __future__ import annotations

import argparse
import json
import re
import time
from dataclasses import fields
from pathlib import Path

from . import spec
from .circuit import design_hash, to_dict, topology_hash
from .evaluate import Evaluation
from .llm import GemmaClient, ScriptedLLM
from .netlist import reference_circuit, witness_circuit
from .prompts import condition_table
from .records import Candidate, RunRecorder
from .report import build_report
from .search import SearchConfig, evaluate_circuit, run_search
from .sim import Simulator, ltspice_backend


# ---------------------------------------------------------------- dry-run stand-ins

class _FakeResult:
    def __init__(self, meas, folder):
        self.ok, self.meas, self.error, self.folder = True, meas, "", folder


def _uses(text: str, model: str) -> bool:
    return re.search(rf"^[DS]\w* .* {model}$", text, re.M) is not None


def analytic_backend(text: str, folder: str, timeout: float):
    """Plausible, deterministic numbers for a dry run (NOT a circuit simulation)."""
    Path(folder).mkdir(parents=True, exist_ok=True)
    (Path(folder) / "circuit.net").write_text(text, encoding="utf-8")
    ip = float(re.search(r"^Ipz pzn pzp SINE\(0 (\S+) ", text, re.M).group(1))
    rl = float(re.search(r"^RL out 0 (\S+)$", text, re.M).group(1))
    boost = 2.5 if _uses(text, "SWI") else 1.0
    v = boost * (ip / 50e-6) * 2.2 * min(1.0, rl / 330e3) ** 0.3
    if _uses(text, "DZ33"):
        v = min(v, 3.35)
    meas = {"vavg": v, "vavg_prev": v - 0.005, "vpp": 0.02, "vmax": v * 1.02,
            "pload": v * v / rl, "pext": 1.3 * v * v / rl}
    if v > 1.85:
        meas["tup"] = 0.2
    return _FakeResult(meas, folder)


def _fake_replies() -> list:
    ref = to_dict(reference_circuit())
    clamp = dict(ref, name="bridge + zener clamp", move="modify_topology")
    clamp["components"] = ref["components"] + [{"id": "D5", "type": "D", "nodes": ["0", "out"], "model": "DZ33"}]
    sshi = dict(to_dict(witness_circuit()), name="p-SSHI + bridge + clamp", move="new_topology")
    dup = json.loads(json.dumps(sshi).replace('"n1"', '"mid"'))
    return [json.dumps(clamp), json.dumps(dup), json.dumps(sshi)]


# ---------------------------------------------------------------- smoke test

def run_smoke(sim: Simulator, rec: RunRecorder) -> None:
    """Reference and witness on all conditions: checks the simulator setup and the spec's difficulty."""
    rec.write_json("spec.json", spec.spec_dict())
    rec.write_json("config.json", {"mode": "smoke"})
    rec.progress(phase="sim", stop_reason=None)
    out = {}
    for k, (circ, origin) in enumerate(((reference_circuit(), "reference"), (witness_circuit(), "witness"))):
        cid = f"c{k:04d}"
        for split, conds in (("search", spec.search_conditions()), ("heldout", spec.heldout_conditions())):
            rec.log("sim", f"{origin}: {len(conds)} {split} conditions")
            t = time.time()
            ev = evaluate_circuit(circ, conds, sim, f"{origin}-{split}")
            rec.log("eval", f"{origin} {split}: {ev.cond_pass}/{len(conds)} conditions pass ({time.time() - t:.1f} s)")
            print(condition_table(ev, conds), flush=True)
            rec.candidate(Candidate(cid, 0, origin, None, circ, topology_hash(circ), design_hash(circ), ev,
                                    origin, circ.name, len(circ.components)), split=split)
            out[(origin, split)] = ev
    ref_s1 = out[("reference", "search")].meas.get("S1", {})
    pext = ref_s1.get("pext")
    if pext is None or pext <= 0:
        rec.log("eval", f"WARNING: reference P_ext on S1 is {pext}; expected > 0. Check the sign of the pext "
                        f".meas in netlist.py before trusting P_ext columns.")
    wit_tup = [m.get("tup") for m in out[("witness", "search")].meas.values() if m.get("tup") is not None]
    if wit_tup and all(abs(t - 1.8) < 1e-9 for t in wit_tup):
        rec.log("eval", "WARNING: every t_up equals 1.8: ltspice_run is reading the threshold, not the crossing "
                        "time, from the 'tup' measurement line. Fix its parsing before the full run.")
    ref_fail = [c for c in spec.search_conditions()
                if not all(ch.passed for ch in out[("reference", "search")].checks if ch.condition == c.id)]
    rec.log("eval", f"reference fails {len(ref_fail)}/6 search conditions: {', '.join(c.id for c in ref_fail)} "
                    f"(the spec is meant to make it fail several)")
    st = sim.stats
    rec.timing({"wall_s": round(rec.elapsed(), 1), "model_s": 0, "model_calls": 0, "prompt_tokens": 0,
                "completion_tokens": 0, "reasoning_tokens": 0, "sim_seconds": round(st["sim_seconds"], 1),
                "sim_wall_s": round(st["wall_seconds"], 1), "n_sims": st["n_sims"], "cache_hits": st["cache_hits"],
                "refine_s": 0})
    rec.write_json("outcome.json", {"stop_reason": "smoke", "met_search": out[("witness", "search")].all_pass,
                                    "met_all": out[("witness", "search")].all_pass and out[("witness", "heldout")].all_pass,
                                    "selected_id": "c0001", "selected_name": witness_circuit().name,
                                    "selected_origin": "witness", "reference_id": "c0000", "failed_checks": []})
    rec.progress(phase="done", stop_reason="smoke")


# ---------------------------------------------------------------- main

def load_config(path) -> SearchConfig:
    if not path:
        return SearchConfig()
    raw = json.loads(Path(path).read_text(encoding="utf-8"))
    known = {f.name for f in fields(SearchConfig)}
    unknown = sorted(set(raw) - known)
    if unknown:
        print(f"WARNING: ignoring unknown config keys: {', '.join(unknown)}", flush=True)
    return SearchConfig(**{k: v for k, v in raw.items() if k in known})


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--out", default=f"runs/{time.strftime('%Y%m%d-%H%M%S')}")
    ap.add_argument("--config")
    ap.add_argument("--smoke", action="store_true", help="reference and witness only, no model")
    ap.add_argument("--fake", action="store_true", help="scripted model + analytic stand-in for LTspice")
    ap.add_argument("--workers", type=int, default=8, help="parallel LTspice simulations")
    ap.add_argument("--sim-timeout", type=float, default=120.0, help="seconds per simulation")
    a = ap.parse_args(argv)
    out = Path(a.out)
    cfg = load_config(a.config)
    if a.fake:
        cfg.heartbeat_s = min(cfg.heartbeat_s, 1.0)
    backend = analytic_backend if a.fake else ltspice_backend()
    sim = Simulator(backend, out, workers=a.workers, timeout=a.sim_timeout)
    rec = RunRecorder(out)
    rec.log("setup", f"output in {out}; mode {'smoke' if a.smoke else 'search'}{' (fake)' if a.fake else ''}; "
                     f"{a.workers} parallel simulations")
    try:
        if a.smoke:
            run_smoke(sim, rec)
        else:
            llm = ScriptedLLM(_fake_replies()) if a.fake else GemmaClient()
            run_search(cfg, llm, sim, rec)
    finally:
        build_report(out)
        rec.log("done", f"report: {out / 'report.md'}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile piezo_synth/reproduce.py
"""Re-simulate a saved circuit on chosen conditions, optionally keeping full waveforms.

    python3 -m piezo_synth.reproduce runs/<id>/selected/circuit.json --out runs/<id>/repro \
        --conditions all --waveforms
"""
from __future__ import annotations

import argparse
import json
from dataclasses import asdict
from pathlib import Path

from . import spec
from .circuit import parse_circuit
from .evaluate import evaluate
from .netlist import build_netlist
from .prompts import condition_table
from .sim import Simulator, ltspice_backend


def pick_conditions(arg: str):
    if arg == "search":
        return spec.search_conditions()
    if arg == "heldout":
        return spec.heldout_conditions()
    if arg == "all":
        return list(spec.CONDITIONS)
    return [spec.condition(c.strip()) for c in arg.split(",") if c.strip()]


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("circuit", help="circuit.json (e.g. runs/<id>/selected/circuit.json)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--conditions", default="all", help="search | heldout | all | comma list such as S1,H2")
    ap.add_argument("--waveforms", action="store_true", help="save all signals, uncompressed .raw")
    ap.add_argument("--fake", action="store_true", help="analytic stand-in for LTspice (dry run)")
    ap.add_argument("--workers", type=int, default=8)
    a = ap.parse_args(argv)
    circ = parse_circuit(json.loads(Path(a.circuit).read_text(encoding="utf-8")))
    conds = pick_conditions(a.conditions)
    if a.fake:
        from .run import analytic_backend
        backend = analytic_backend
    else:
        backend = ltspice_backend()
    sim = Simulator(backend, Path(a.out), workers=a.workers)
    jobs = []
    for cond in conds:
        text = build_netlist(circ, cond, title=f"reproduce {circ.name}", save_all=a.waveforms)
        if a.waveforms:
            text = text.replace("\n.end\n", "\n.options plotwinsize=0\n.end\n")
        jobs.append((cond.id, text))
    results = sim.run(jobs)
    ev = evaluate({c.id: r for c, r in zip(conds, results)}, conds)
    print(condition_table(ev, conds), flush=True)
    waves = {}
    for cond, r in zip(conds, results):
        raws = sorted(Path(r.folder).glob("*.raw"))
        waves[cond.id] = str(raws[0]) if raws else None
        if not r.ok:
            print(f"{cond.id}: simulation failed: {r.error[:300]}")
    (Path(a.out) / "reproduce.json").write_text(json.dumps({
        "circuit": a.circuit, "conditions": [c.id for c in conds], "all_pass": ev.all_pass,
        "checks": [asdict(c) for c in ev.checks], "meas": ev.meas, "raw_files": waves}, indent=1), encoding="utf-8")
    print(f"all pass: {ev.all_pass}; details in {Path(a.out) / 'reproduce.json'}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 3. Settings

- `max_proposals`: model proposals at most (each ~0.5-2.5 min with thinking on).
- `min_proposals`: the model may only declare itself done after this many.
- `wall_clock_s`: stop starting new proposals after this many seconds.
- `stagnation`: stop after this many proposals without improving the best score.
- `refine_budget`: value-tuning evaluations per tuning pass (each = 6 parallel simulations).
- `think`: Gemma reasoning on for proposals (recommended).
- `seed`: random seed for the value tuner (the model itself is not deterministic).

In [ ]:
CONFIG = dict(max_proposals=16, min_proposals=4, wall_clock_s=3600, stagnation=6,
              refine_budget=10, refine_min=3, think=True, seed=1)
Path("run_config.json").write_text(json.dumps(CONFIG, indent=1))
CONFIG

## 4. Smoke test (first run only)

Expect: 20 simulations, `[sim]`/`[eval]` lines and two tables per circuit. Check that the reference
**fails** several search conditions (that is what makes the spec challenging) and that the witness
passes more of them. If the job prints a WARNING about `P_ext`, read the README section "First run".

In [ ]:
smoke = run("bash -o pipefail -c 'rm -rf runs/smoke && mkdir -p runs && python3 -u -m piezo_synth.run --smoke --out runs/smoke "
            "2>&1 | tee runs/smoke.log'",
            image=LTSPICE, gpus=0, cpus=8, memory="16Gi", timeout=3600, name="piezo-smoke")
log(smoke)

## 5. Full experiment

One CPU job runs everything: Gemma calls (served on the cluster's GPUs), LTspice simulations in 8
parallel threads, value tuning while Gemma thinks, the held-out check and the report. The console
output is also saved to `runs/<RUN_ID>.log` (the job's own log is deleted after 48 hours).

In [ ]:
job = run(f"bash -o pipefail -c 'mkdir -p runs && python3 -u -m piezo_synth.run --config run_config.json "
          f"--out runs/{RUN_ID} 2>&1 | tee runs/{RUN_ID}.log'",
          image=LTSPICE, gpus=0, cpus=8, memory="16Gi", timeout=4 * 3600, name="piezo-synth")
print("submitted", job, "-> runs/" + RUN_ID)
log(job)

## 6. Progress, results, figures

To look at an earlier run, set `RUN_ID` to its folder name under `runs/` first.

In [ ]:
RUN = Path("runs") / RUN_ID
p = json.loads((RUN / "progress.json").read_text()) if (RUN / "progress.json").exists() else {}
print(f"phase: {p.get('phase')}   proposal {p.get('iteration')}/{p.get('max_proposals')}   "
      f"elapsed {p.get('elapsed_s', 0) / 60:.1f} min")
print(f"best so far: {p.get('best_id')} {p.get('best_name')!r}  score {p.get('best_scalar')}  "
      f"conditions passed {p.get('best_cond_pass')}/6")
if p.get("phase") == "model":
    print(f"waiting for the model for {p.get('model_wait_s', 0)} s")
if p.get("stop_reason"):
    print("stop reason:", p["stop_reason"])

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
from piezo_synth.report import build_report

build_report(RUN)          # refresh (also works on an unfinished or killed run)
display(Markdown((RUN / "report.md").read_text()))
cands = pd.read_csv(RUN / "candidates.csv")
meas = pd.read_csv(RUN / "measurements.csv")
checks = pd.read_csv(RUN / "checks.csv")
summary = json.loads((RUN / "summary.json").read_text())
SEL, REF = summary["selected_id"], summary["reference_id"]
cands.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from piezo_synth import spec

figdir = RUN / "figures"; figdir.mkdir(exist_ok=True)
search = cands[cands.split == "search"].copy()

# 1. score over the search, proposals coloured by move, tuning steps in grey
fig, ax = plt.subplots(figsize=(8, 4))
search["order"] = range(len(search))
best = search.scalar.cummax()
ax.plot(search.order, best, "k-", lw=1.5, label="best so far")
colours = {"new_topology": "tab:blue", "modify_topology": "tab:orange", "resize": "tab:green",
           "refine": "lightgrey", "reference": "tab:red"}
for mv, grp in search.groupby("move"):
    ax.scatter(grp.order, grp.scalar, s=25 if mv != "refine" else 10, c=colours.get(mv, "k"), label=mv, zorder=3)
ax.set(xlabel="design evaluated (in order)", ylabel="score", title="Search progress")
ax.legend(fontsize=8, ncol=3); fig.tight_layout()
fig.savefig(figdir / "search_progress.png", dpi=200); fig.savefig(figdir / "search_progress.pdf")

# 2. pass/fail map: selected vs reference, all conditions x criteria
conds = [c.id for c in spec.CONDITIONS]; crits = [c.id for c in spec.CRITERIA]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, (cid, title) in zip(axes, [(SEL, f"selected ({SEL})"), (REF, f"reference ({REF})")]):
    sub = checks[checks.candidate == cid]
    grid = np.full((len(crits), len(conds)), np.nan)
    for _, r in sub.iterrows():
        grid[crits.index(r.criterion), conds.index(r.condition)] = 1.0 if str(r.passed) == "True" else 0.0
    ax.imshow(grid, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(conds)), conds); ax.set_yticks(range(len(crits)), crits); ax.set_title(title)
    ax.axvline(5.5, color="k", lw=1)          # left: search conditions, right: held-out
fig.suptitle("green = pass, red = fail; conditions right of the line are held out"); fig.tight_layout()
fig.savefig(figdir / "pass_fail.png", dpi=200); fig.savefig(figdir / "pass_fail.pdf")

# 3. output voltage and load power per condition
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
x = np.arange(len(conds)); w = 0.38
for k, (cid, lab) in enumerate([(SEL, "selected"), (REF, "reference")]):
    sub = meas[meas.candidate == cid].set_index("condition").reindex(conds)
    axes[0].bar(x + (k - 0.5) * w, sub.vavg, w, label=lab)
    axes[1].bar(x + (k - 0.5) * w, sub.pload * 1e6, w, label=lab)
for v in (1.8, 3.6):
    axes[0].axhline(v, color="k", ls="--", lw=0.8)
axes[0].set(ylabel="mean V(out), V", xticks=x, xticklabels=conds)
axes[1].set(ylabel="load power, uW", xticks=x, xticklabels=conds)
axes[0].legend(); fig.tight_layout()
fig.savefig(figdir / "vout_power.png", dpi=200); fig.savefig(figdir / "vout_power.pdf")
print("figures saved in", figdir)

### Waveforms

Re-simulates the selected design and the reference on S1 and S4 with all signals saved, then plots
V(out). This is also how to reproduce any design: point `reproduce` at its `circuit.json`.

In [ ]:
wave = run(f"python3 -u -m piezo_synth.reproduce runs/{RUN_ID}/selected/circuit.json "
           f"--out runs/{RUN_ID}/waves-selected --conditions S1,S4 --waveforms && "
           f"python3 -u -m piezo_synth.reproduce runs/{RUN_ID}/candidates/{REF}/circuit.json "
           f"--out runs/{RUN_ID}/waves-reference --conditions S1,S4 --waveforms",
           image=LTSPICE, gpus=0, cpus=4, memory="8Gi", timeout=1800, name="piezo-waves")
log(wave)

In [ ]:
from piezo_synth.rawread import read_raw

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, cond in zip(axes, ["S1", "S4"]):
    for lab in ("selected", "reference"):
        info = json.loads((RUN / f"waves-{lab}" / "reproduce.json").read_text())
        raw = info["raw_files"].get(cond)
        if raw:
            d = read_raw(raw)
            ax.plot(d["time"], d["V(out)"], lw=0.8, label=lab)
    for v in (1.8, 3.6):
        ax.axhline(v, color="k", ls="--", lw=0.6)
    ax.set(title=cond, xlabel="time, s", ylabel="V(out), V"); ax.legend()
fig.tight_layout(); fig.savefig(figdir / "waveforms.png", dpi=200); fig.savefig(figdir / "waveforms.pdf")